# The k-means assign kernel: what changed, and what holds up

Run top to bottom on an **A100** (Runtime > Change runtime type). It takes about 25 minutes.

Claims you can check here:
1. **iter06** (phase 1) is about 1.36× faster than the original on real keys and passes the strict correctness test
   on real and held-out keys. A third agent pass seeded with iter06 found nothing faster.
2. The **int8** kernel is 1.74× faster on synthetic keys and passes phase 1's test there, but fails the strict test.
3. **Phase 2's** 2.4× came from key files that skipped a step of Double-P's pipeline; on real keys it is slower
   than iter06.
4. The harness: timing in one process, the repeat-call check, and clock drift between processes.
5. Where the rest of iter06's speedup, and its fewer atomic sectors, come from (section 7).
6. Where both kernels sit against the A100's peak math rate and memory bandwidths (section 8, roofline).

Needs, in Google Drive under `MyDrive/kernel-agent/`: `centered/real_keys_gate.pt`, `centered/real_keys_heldout.pt`
(made by `test.ipynb`), `runs/20261026-004515/iter06.py`, `runs/20261026-004515/best_kernel.py` (int8) and
`runs/20261026-033411/best_kernel.py` (phase 2).

In [ ]:
# Which Triton compiles the kernels. Colab ships 3.6.0. Triton 3.8.0 and later cap FP32 atomic adds at one element per
# thread before Hopper (PR #10059), the layout change section 7 traces. To rerun on 3.8.0 or later: uncomment the pip line, run
# this cell, then Runtime > Restart session and run from the top. pip will warn that PyTorch pins another Triton; that
# only matters for torch.compile, which this notebook doesn't use.
# !pip install -q triton==3.8.0
import triton
print("triton", triton.__version__)

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/kernel-agent"
copies = {"k_iter06.py": f"{D}/runs/20261026-004515/iter06.py",
          "k_int8.py": f"{D}/runs/20261026-004515/best_kernel.py",
          "k_phase2.py": f"{D}/runs/20261026-033411/best_kernel.py"}
for p in [f"{D}/centered/real_keys_gate.pt", f"{D}/centered/real_keys_heldout.pt", *copies.values()]:
    print("ok     " if os.path.exists(p) else "MISSING", p)
for dst, src in copies.items():
    if os.path.exists(src):
        shutil.copy(src, f"/content/{dst}")

## 1. What the kernel does

One step of k-means on attention keys. For each key (128 numbers), find the centroid with the highest dot product,
then add the key into that centroid's running sum `s` and add 1 to its count `cnt`. The next step divides `s` by `cnt`
to get the new centroids.

Shapes: 8 rows, each one (layer, head) of the model; 32,768 keys per row; 2,048 centroids; 128 dims. Each Triton
program takes 128 keys of one row and loops over the centroids 32 at a time. It scores them on tensor cores with
`tl.dot`, keeps the best centroid per key, and finishes with the atomic adds.

| Keys | What they are | Used for |
|---|---|---|
| synthetic | random Gaussian keys, random unit-length centroids | phase 1's only test and timing data |
| gate keys | real Qwen2.5-1.5B keys from one 32,836-token prompt (Pride and Prejudice from its start), prepared the way Double-P clusters them (below); 8 (layer, head) rows (layers 0, 3, 7, …, 27); centroids from Double-P's segment k-means | the third pass's test and timing data (the agent saw these) |
| held-out keys | same method, a prompt from book token 80,000, other (layer, head) pairs (layers 0, 2, 6, …, 26) | checked only after the run (the agent never saw them) |

**How Double-P prepares keys, and what the earlier files got wrong.** Double-P clusters keys once, at prefill
(`RetroClusterIndex.build` in `sparse_attention/index/cluster/retro_cluster.py`). It leaves out the first 4 tokens
(attention sinks) and the last 64 (recent window), which it keeps exact, and it subtracts each head's mean key before
k-means. The key files phases 1 and 2 used skipped both steps. That mattered on layer 0: Qwen2.5's key projection adds
a bias, and in layer 0 that bias is as long as the keys themselves (|bias| ≈ 821 against a median |key| ≈ 821 in
head 0), so raw layer-0 keys all point nearly the same way. Up to 99% of them landed in one cluster, and all their
atomic adds hit the same addresses. The original kernel was very slow on that input, which inflated every layer-0
speedup in the earlier runs. Subtracting the mean removes the bias. In the files used here the biggest cluster holds
0.1–0.3% of keys in every row, layer 0 included.

The original kernel is below; read this one. The next cells are helpers you can run without reading.

In [ ]:
%%writefile k_original.py
"""The original assign kernel from Double-P (sparse_attention/index/cluster/kmeans.py, _triton_assign_kernel).

One program handles BLOCK_N keys of one batch row: score them against every centroid with tensor cores, keep the best
centroid per key, then add each key into its centroid's running sum (s) and count (cnt) with atomics.
"""

import triton
import triton.language as tl


@triton.jit
def _assign(
    K, X, S, C, M,
    stride_kz: tl.constexpr, stride_kn: tl.constexpr, stride_kd: tl.constexpr,
    stride_xz: tl.constexpr, stride_xk: tl.constexpr, stride_xd: tl.constexpr,
    stride_sz: tl.constexpr, stride_sk: tl.constexpr, stride_sd: tl.constexpr,
    stride_cz: tl.constexpr, stride_ck: tl.constexpr,
    stride_mz: tl.constexpr, stride_mn: tl.constexpr,
    num_tokens: tl.constexpr, num_centroids: tl.constexpr,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr,
):
# PROLOGUE of Tile processing
    start_n = tl.program_id(0) * BLOCK_N
    batch_idx = tl.program_id(1)
    if start_n >= num_tokens:
        return
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    n_mask = offs_n < num_tokens
    k_ptrs = K + batch_idx * stride_kz + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    x_ptrs = X + batch_idx * stride_xz + offs_k[None, :] * stride_xk + offs_d[:, None] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz
    m_ptrs = M + batch_idx * stride_mz + offs_n * stride_mn

    # start of Main Loop
    k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)  # [BLOCK_N, D] keys, loaded once
    max_val = tl.zeros([BLOCK_N], dtype=tl.float32) - float("inf")
    max_idx = tl.zeros([BLOCK_N], dtype=tl.int32) 

    for start_k in tl.range(0, num_centroids, BLOCK_K):
        x = tl.load(x_ptrs)  # [D, BLOCK_K] centroids
        ip = tl.dot(k, x).to(tl.float32)  # [k space, centroid space] block K is centroids, block N is k
        tmp_max_val, tmp_max_idx = tl.max(ip, axis=1, return_indices=True) # get max centroid idx, with max score value
        tmp_max_idx += start_k # increment tmp_max_ids to reflect the block offset in the centroid space
        max_idx = tl.where(tmp_max_val > max_val, tmp_max_idx, max_idx) # elemenwise if condition for each 1D value of K, update max_idx if tmp_max_val is greater
        max_val = tl.maximum(tmp_max_val, max_val) # elementwise maximum of the current max and the new max values
        x_ptrs += BLOCK_K * stride_xk # move the pointer to the next block of centroids
    
    # EPILOGUE of Tile processing
    tl.store(m_ptrs, max_idx, mask=n_mask) #[BLOCK_N] max_idx
    # s[label[n], :] += k[n, :] for the whole [BLOCK_N, D] tile in one statement. The atomics use the register layout
    # Triton chose for loading k and feeding the tensor cores, which is not arranged for atomics (see the notebook).
    # [block_n, dim] atomic add
    # actually the matrix is [k projection, centroid space] and they are memory addresses of a centroid vector to 
    # write a tile of k into.
    # each row in that k tile corresponds to a centroid address
    # we can have many same centroid addresses in the k tile
    # tl.atomic_add is read, modify, write where modify means += and read,modify,write means a single atomic operation. it cannot be interrupted mid operation.  This avoids race conditions and concurrency issues.
    # it is at the cost of performance, as now operations are serialized and cannot be overlapped.
    # internally each element of the tile is processed one at a time.  This goes from O(1) elementwise ops to O(BLOCK_N * D) atomic operations.
    # s_ptrs just gets you in the ballpark
    # max_idx contains the index of the centroid with the maximum score for each row in the k tile
    # strid_sk gives you row stride for each index in the k tile ( SO WE CAN FILL IN THE ELEMENTS OF EACH VECTOR IN THE ROW)

    # notice we are passing 2D matrix to atomic_add
    tl.atomic_add(s_ptrs + max_idx[:, None] * stride_sk, k.to(tl.float32), mask=n_mask[:, None], sem="relaxed")
    # cnt[label[n]] += 1
    # this is adding count vectors for later component wise scalar division [1,1,1,...] 
    # shape [BLOCK_N]
    tl.atomic_add(c_ptrs + max_idx * stride_ck, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed") 


def launch(data, cent, s, cnt, m):
    """data [B, N, D] bf16, cent [B, C, D] bf16 (unit norm), s [B, C, D] fp32 and cnt [B, C] int32 zeroed, m [B, N] int32."""
    b, n, d = data.shape
    c = cent.shape[1]
    _assign[(triton.cdiv(n, 128), b, 1)](
        data, cent, s, cnt, m,
        *data.stride(), *cent.stride(), *s.stride(), *cnt.stride(), *m.stride(),
        n, c,
        BLOCK_N=128, BLOCK_K=32, BLOCK_D=d, num_warps=4, num_stages=2,  # the config kmeans.py uses
    )

In [ ]:
%%writefile k_ablation.py
"""Ablation: the original kernel with iter06's two changes switched on one at a time.

Written for this notebook (not by the agent), from the agent's code, with the original's tile sizes (128 keys x 32
centroids, 4 warps, 2 stages) so that only the two changes differ:

    PACKED       False: argmax as in the original (max + index search + select per tile)
                 True:  pack the centroid index into the low bits of the fp32 score, so the argmax is a single max
                 2:     defer the argmax like True, without packing: keep a [keys x centroids] tile of best scores and
                        one of their indices, update both elementwise, and reduce once after the loop (exact)
    EPI          0: atomics straight from the key tile, as in the original
                 1: re-read the keys in a flat layout, one key row after another (compiler can see the addresses are
                    contiguous)
                 2: same as 1, but the column offset is multiplied by a runtime 1 (stride_one) that the compiler cannot
                    see through, as the agent did
                 3: atomics straight from the key tile, with the column offset multiplied by stride_one (no reload)
                 4: same as 1, but only the atomics' column offset is multiplied by stride_one
                 5: same as 1, but only the loads' column offset is multiplied by stride_one
    REDUCE_EACH  (packed only) False: keep a [keys x centroids] tile of packed maxima and reduce it once after the loop
                 True:  reduce each tile to one value per key inside the loop

variant() also takes tile sizes, warps and stages, so section 7 can run variant 5 with iter06's launch settings, and
maxnreg, a cap on registers per thread that Triton passes to ptxas.
"""

import triton
import triton.language as tl


@triton.jit(do_not_specialize=["stride_one"])
def _assign(
    K, X, S, CNT, M, N, C, stride_one,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, D: tl.constexpr,
    MASK_HI: tl.constexpr, MASK_LO: tl.constexpr,
    PACKED: tl.constexpr, EPI: tl.constexpr, EPI_ROWS: tl.constexpr, REDUCE_EACH: tl.constexpr,
):
    pid_n = tl.program_id(0)
    b = tl.program_id(1).to(tl.int64)
    start_n = pid_n * BLOCK_N
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_d = tl.arange(0, D)
    offs_k = tl.arange(0, BLOCK_K)
    n_mask = offs_n < N
    k = tl.load(K + b * N * D + offs_n[:, None] * D + offs_d[None, :], mask=n_mask[:, None], other=0.0)
    x_ptrs = X + b * C * D + offs_k[None, :] * D + offs_d[:, None]

    if PACKED == 2:
        # best score and its centroid index for every [key, column] slot, reduced once after the loop
        best = tl.full([BLOCK_N, BLOCK_K], float("-inf"), tl.float32)
        best_idx = tl.zeros([BLOCK_N, BLOCK_K], dtype=tl.int32)
        for start in range(0, C, BLOCK_K):
            ip = tl.dot(k, tl.load(x_ptrs)) # [BLOCK_N, BLOCK_K] scores for this tile ofs centroids
            better = ip > best
            best_idx = tl.where(better, start + offs_k[None, :], best_idx)
            best = tl.where(better, ip, best)
            x_ptrs += BLOCK_K * D
        row_best = tl.max(best, axis=1)
        max_idx = tl.min(tl.where(best == row_best[:, None], best_idx, C), axis=1)  # lowest index among ties
    elif PACKED:
        # score bits with the low IDX_BITS cleared, OR the centroid index: comparing the packed floats compares
        # scores (to 12 fraction bits for 2048 centroids) and carries the winner's index along for free
        if REDUCE_EACH:
            best = tl.full([BLOCK_N], float("-inf"), tl.float32)
        else:
            best = tl.full([BLOCK_N, BLOCK_K], float("-inf"), tl.float32)
        for start in range(0, C, BLOCK_K):
            ip = tl.dot(k, tl.load(x_ptrs))
            bits = ip.to(tl.int32, bitcast=True)
            idx = start + offs_k
            packed = ((bits & MASK_HI) | idx[None, :]).to(tl.float32, bitcast=True)
            if REDUCE_EACH:
                best = tl.maximum(best, tl.max(packed, axis=1))
            else:
                best = tl.maximum(best, packed)
            x_ptrs += BLOCK_K * D
        if REDUCE_EACH:
            row_best = best
        else:
            row_best = tl.max(best, axis=1)
        max_idx = row_best.to(tl.int32, bitcast=True) & MASK_LO
    else:
        max_val = tl.full([BLOCK_N], float("-inf"), tl.float32)
        max_idx = tl.zeros([BLOCK_N], dtype=tl.int32)
        for start in range(0, C, BLOCK_K):
            ip = tl.dot(k, tl.load(x_ptrs))
            v, i = tl.max(ip, axis=1, return_indices=True)
            max_idx = tl.where(v > max_val, i + start, max_idx)
            max_val = tl.maximum(v, max_val)
            x_ptrs += BLOCK_K * D

    m_base = M + b * N
    tl.store(m_base + offs_n, max_idx, mask=n_mask)
    tl.atomic_add(CNT + b * C + max_idx, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed")
    if EPI == 0:
        tl.atomic_add(S + b * C * D + max_idx[:, None] * D + offs_d[None, :], k.to(tl.float32),
                      mask=n_mask[:, None], sem="relaxed")
    elif EPI == 3:  # tile atomics as in the original, but the compiler can't see that the columns are contiguous
        tl.atomic_add(S + b * C * D + max_idx[:, None] * D + (offs_d * stride_one)[None, :], k.to(tl.float32),
                      mask=n_mask[:, None], sem="relaxed")
    else:
        tl.debug_barrier()  # the labels stored above must be visible to every thread before they are re-read
        ii = tl.arange(0, EPI_ROWS * D)
        r_loc = ii // D  # which key row
        cc = ii % D  # which column: consecutive threads get consecutive columns of the same row
        if EPI == 2:
            cc = cc * stride_one
        c_load = cc
        c_add = cc
        if EPI == 4:  # hide the contiguity from the atomics only
            c_add = cc * stride_one
        if EPI == 5:  # hide it from the loads only
            c_load = cc * stride_one
        for r0 in range(0, BLOCK_N, EPI_ROWS):
            rr = start_n + r0 + r_loc
            rmask = rr < N
            lab = tl.load(m_base + rr, mask=rmask, other=0)
            kv = tl.load(K + b * N * D + rr * D + c_load, mask=rmask, other=0.0)  # re-read keys (from L2)
            tl.atomic_add(S + b * C * D + lab * D + c_add, kv.to(tl.float32), mask=rmask, sem="relaxed")


def variant(packed, epi, reduce_each=False, block_n=128, block_k=32, epi_rows=32, num_warps=4, num_stages=2,
            maxnreg=None):
    cap = {"maxnreg": maxnreg} if maxnreg else {}  # only when capping, so the other variants compile as before

    def launch(data, cent, s, cnt, m):
        b, n, d = data.shape
        c = cent.shape[1]
        bits = max(1, (c - 1).bit_length())
        _assign[(triton.cdiv(n, block_n), b, 1)](
            data, cent, s, cnt, m, n, c, 1,
            BLOCK_N=block_n, BLOCK_K=block_k, D=d, MASK_HI=-(1 << bits), MASK_LO=(1 << bits) - 1,
            PACKED=packed, EPI=epi, EPI_ROWS=min(epi_rows, block_n), REDUCE_EACH=reduce_each,
            num_warps=num_warps, num_stages=num_stages, **cap,
        )
    return launch

In [ ]:
%%writefile lab.py
"""Helpers for the notebook: inputs, kernels, correctness checks and same-process timing."""

import importlib.util
import json
import statistics
from pathlib import Path

import torch
import triton

DEV = "cuda"
HERE = Path(__file__).resolve().parent
DRIVE = "/content/drive/MyDrive/kernel-agent"
GATE, HELDOUT = f"{DRIVE}/centered/real_keys_gate.pt", f"{DRIVE}/centered/real_keys_heldout.pt"
ABS_TOL = 0.05  # phase-1 rule: chosen centroid's score within 0.05 of the best (absolute)
REL_TOL = 2.0 ** -12  # phase-2 rule: within 2^-12 x |key| of the best, i.e. a cosine gap (centroids are unit norm)
torch.backends.cuda.matmul.allow_tf32 = False  # the reference must be real fp32

FILES = {"original": "k_original.py", "iter06": "k_iter06.py", "int8": "k_int8.py", "phase2": "k_phase2.py"}
ABLATION = {  # name: (PACKED, EPI), see k_ablation.py
    "plain argmax, tile atomics": (False, 0),
    "packed argmax, tile atomics": (True, 0),
    "plain argmax, flat atomics": (False, 2),
    "packed, flat (stride visible)": (True, 1),
    "packed, flat (iter06 style)": (True, 2),
}
TILE_STRIDE = {  # section 7: variants 1 and 2 with the tile atomic's column offset times stride_one
    "plain, tile, runtime stride": (False, 3),
    "packed, tile, runtime stride": (True, 3),
}
DEFERRED = {  # section 7: variant 3's epilogue with the argmax deferred like iter06's, but without packing
    "deferred, unpacked, flat atomics": (2, 2),
}
STRIDE_SPLIT = {  # section 7: v4 and v5 differ only in stride_one; these hide it from one side of the epilogue
    "packed, flat (hidden: atomics)": (True, 4),
    "packed, flat (hidden: loads)": (True, 5),
}
REG_CAP = {  # section 7: v4 and v5 with a register cap, to match blocks per SM. name: (PACKED, EPI, maxnreg)
    "v4, at most 168 registers": (True, 1, 168),
    "v5, at most 128 registers": (True, 2, 128),
}


def load_kernel(path):
    spec = importlib.util.spec_from_file_location(Path(path).stem, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.launch


def int8_ncand(n):
    """The int8 kernel limited to its configs that rescore n candidates (its own tuner picks among 2, 3 and 4)."""
    spec = importlib.util.spec_from_file_location(f"k_int8_x{n}", str(HERE / FILES["int8"]))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod._CONFIGS = [cfg for cfg in mod._CONFIGS if cfg[0] == "i8" and cfg[-1] == n]
    assert mod._CONFIGS, f"k_int8.py has no int8 config with {n} candidates"

    def launch(data, cent, s, cnt, m):
        mod.launch(data, cent, s, cnt, m)
        # if none of them compiles, its tuner falls back to a bf16 config
        assert all(cfg[0] == "i8" for cfg in mod._CACHE.values()), f"no int8 config with {n} candidates ran"
    return launch


def kernels():
    return {**{name: load_kernel(HERE / f) for name, f in FILES.items()}, "int8 x4": int8_ncand(4)}


def ablation():
    spec = importlib.util.spec_from_file_location("k_ablation", str(HERE / "k_ablation.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = {name: mod.variant(*flags) for name, flags in ABLATION.items()}
    path = HERE / "iter06_config.json"  # iter06's launch settings, saved by section 7
    if path.exists():
        cfg = json.loads(path.read_text())
        reduce_each = cfg.pop("reduce_each")
        out["v5 + iter06 settings"] = mod.variant(True, 2, **cfg)
        if reduce_each:
            out["v5 + iter06 settings + reduce"] = mod.variant(True, 2, reduce_each=True, **cfg)
    out.update({name: mod.variant(*flags) for name, flags in {**TILE_STRIDE, **DEFERRED, **STRIDE_SPLIT}.items()})
    out.update({name: mod.variant(packed, epi, maxnreg=r) for name, (packed, epi, r) in REG_CAP.items()})
    return out


def synthetic(b=8, n=32768, c=2048, d=128, seed=0):
    """Random Gaussian keys and random unit centroids: no structure at all."""
    g = torch.Generator(device=DEV).manual_seed(seed)
    keys = torch.randn(b, n, d, device=DEV, generator=g).bfloat16()
    cent = torch.randn(b, c, d, device=DEV, generator=g)
    rows = [("row", i) for i in range(b)]
    return keys, (cent / cent.norm(dim=-1, keepdim=True)).bfloat16(), rows


def real(path):
    """Real Qwen2.5-1.5B keys [8, 32768, 128] bf16, prepared as Double-P clusters them (sink and window tokens dropped,
    mean subtracted), their segment k-means centroids [8, 2048, 128], (layer, head) per row."""
    d = torch.load(path, map_location="cpu")
    return d["keys"].to(DEV), d["cent"].to(DEV), [tuple(r) for r in d["rows"]]


def one_row(keys, cent, i):
    """Row i copied into every batch slot: same shape (so each kernel keeps its config), one (layer, head)."""
    b = keys.shape[0]
    return keys[i:i + 1].expand(b, -1, -1).contiguous(), cent[i:i + 1].expand(b, -1, -1).contiguous()


def inputs(which):
    """synthetic | normal (gate row layer 3 head 1), each [8, 32768, 128]."""
    if which == "synthetic":
        return synthetic()[:2]
    keys, cent, _ = real(GATE)
    return one_row(keys, cent, {"normal": 1}[which])


def alloc(keys, cent):
    b, n, d = keys.shape
    c = cent.shape[1]
    return (torch.zeros(b, c, d, device=DEV), torch.zeros(b, c, device=DEV, dtype=torch.int32),
            torch.full((b, n), -1, device=DEV, dtype=torch.int32))


def run(launch, keys, cent):
    s, cnt, m = alloc(keys, cent)
    launch(keys, cent, s, cnt, m)
    torch.cuda.synchronize()
    return s, cnt, m


def check(keys, cent, s, cnt, m):
    """Per row, against exact fp32 math: label agreement, worst score gap (absolute, and divided by |key|), and whether
    the sums and counts are exactly what the kernel's own labels imply."""
    out = []
    for i in range(keys.shape[0]):
        k, x, lab = keys[i].float(), cent[i].float(), m[i].long()
        if int(lab.min()) < 0 or int(lab.max()) >= x.shape[0]:
            out.append({"agree": 0.0, "abs_gap": float("inf"), "cos_gap": float("inf"), "sums_ok": False})
            continue
        sc = k @ x.T  # [N, C] exact scores
        gap = sc.max(1).values - sc.gather(1, lab[:, None]).squeeze(1)
        s_ref = torch.zeros(x.shape, device=DEV, dtype=torch.float64).index_add_(0, lab, k.double())
        cnt_ref = torch.bincount(lab, minlength=x.shape[0])
        sum_err = ((s[i].double() - s_ref).abs().max() / s_ref.abs().max()).item()
        out.append({"agree": (lab == sc.argmax(1)).double().mean().item(), "abs_gap": gap.max().item(),
                    "cos_gap": (gap / k.norm(dim=1).clamp_min(1e-30)).max().item(),
                    "sums_ok": bool(torch.equal(cnt[i].long(), cnt_ref)) and sum_err < 1e-5})
    return out


def rule1(r):
    return r["sums_ok"] and r["abs_gap"] < ABS_TOL


def rule2(r):
    return r["sums_ok"] and r["cos_gap"] <= REL_TOL


def compare(kernels, keys, cent, rounds=9):
    """Time every kernel in this one process, in rounds: each round times each kernel once (order rotated), and a
    kernel's score is the median over rounds of (original time / its time), so clock drift cancels. Each call zeroes
    s and cnt first, as a real k-means iteration does; zeroing alone is timed too and subtracted.
    Returns {name: (median kernel ms, median speedup vs "original", (lowest, highest) per-round speedup)}."""
    s, cnt, m = alloc(keys, cent)

    def make(launch):
        def f():
            s.zero_()
            cnt.zero_()
            launch(keys, cent, s, cnt, m)
        return f

    fns = {name: make(launch) for name, launch in kernels.items()}
    fns["_zero"] = lambda: (s.zero_(), cnt.zero_())
    for f in fns.values():  # compile (and autotune) before timing
        f()
    triton.testing.do_bench(next(iter(fns.values())), warmup=500, rep=500)  # keep the GPU busy so clocks settle
    names = list(fns)
    raw = {k: [] for k in names}
    for r in range(rounds):
        for k in names[r % len(names):] + names[:r % len(names)]:
            raw[k].append(triton.testing.do_bench(fns[k], warmup=25, rep=100))
    zero = raw.pop("_zero")
    t = {k: [a - z for a, z in zip(v, zero)] for k, v in raw.items()}
    out = {}
    for k, ts in t.items():
        ratios = sorted(o / x for o, x in zip(t["original"], ts))
        out[k] = (statistics.median(ts), statistics.median(ratios), (ratios[0], ratios[-1]))
    return out


def show(res):
    for name, (ms, x, (lo, hi)) in res.items():
        print(f"  {name:32s} {ms:.3f} ms  {x:.3f}x vs original (rounds {lo:.3f}-{hi:.3f})")

In [ ]:
%%writefile prof.py
"""One measured launch of a kernel, for Nsight Compute. Run under: ncu --profile-from-start off ... python prof.py ...

    python prof.py KERNEL INPUT [CONFIG]  # KERNEL: original | iter06 | int8 | phase2 | an ablation name
                                          # INPUT: synthetic | normal
                                          # CONFIG: pin iter06's tuner, e.g. 128,64,4,2,0 (BLOCK_N, BLOCK_K,
                                          #   warps, stages, REDUCE_EACH); without it, iter06 tunes in this process
"""

import sys

import torch

import lab

name, which = sys.argv[1], sys.argv[2]
launch = {**lab.kernels(), **lab.ablation()}[name]
keys, cent = lab.inputs(which)
if len(sys.argv) > 3:  # seed the tuner's cache, so launch() runs this config instead of timing all of them
    *ints, last = (int(v) for v in sys.argv[3].split(","))
    (b, n, d), c = keys.shape, cent.shape[1]
    launch.__globals__["_CACHE"][(b, n, c, d)] = (*ints, bool(last))
s, cnt, m = lab.alloc(keys, cent)
launch(keys, cent, s, cnt, m)  # compile, autotune and warm up outside the profiled range
torch.cuda.synchronize()
s.zero_()
cnt.zero_()
torch.cuda.synchronize()
torch.cuda.profiler.start()  # ncu profiles only the kernels launched between start and stop
launch(keys, cent, s, cnt, m)
torch.cuda.synchronize()
torch.cuda.profiler.stop()
print(f"profiled {name} on {which}")

In [ ]:
%%writefile drift.py
"""Time the original kernel on synthetic keys in this fresh process, the way the first agent runs timed each candidate."""

import statistics

import triton

import lab

keys, cent, _ = lab.synthetic()
s, cnt, m = lab.alloc(keys, cent)
launch = lab.load_kernel(lab.HERE / "k_original.py")


def f():
    s.zero_()
    cnt.zero_()
    launch(keys, cent, s, cnt, m)


f()
print(statistics.median(triton.testing.do_bench(f, warmup=25, rep=100) for _ in range(3)))

## 2. The four kernels and what changed

### Atomic adds
After it chooses a centroid for every key, the kernel does `s[label] += key` (128 floats) and `cnt[label] += 1`.
Thousands of threads update `s` at the same time, so these are **atomic** adds: the L2 cache does each
read-add-write as one indivisible step, so no update is lost. `sem="relaxed"` promises nothing beyond that, so the
order of the additions (and the last bits of `s`) varies from run to run.

Two things make atomic adds slow:
- **Coalescing.** A warp is 32 threads that issue one instruction together, and memory works in 32-byte sectors
  (8 floats). If the 32 threads touch 32 neighbouring floats, the instruction needs 4 sectors. If they're spread out,
  it can need up to 32, which is up to 8× the L2 work for the same data. *Coalesced* means neighbouring threads
  touch neighbouring addresses.
- **Collisions.** Two adds to the same float can't happen at once; the L2 does them one after the other. When many
  keys pick the same centroid at the same time, their adds queue up on the same 128 floats. That is what happened on
  the uncentered layer-0 rows. On real keys it happens a little: neighbouring tokens often share a centroid, so a
  tile of 128 real keys picks only 9–23 distinct centroids, against about 124 for random keys.

### original
- **Argmax:** every loop iteration does `tl.max(..., return_indices=True)`, then a compare and a select.
- **Atomics:** one `tl.atomic_add` over the whole [128 keys × 128 dims] tile, in the register layout Triton chose
  for loading `k` and feeding the tensor cores. That layout is arranged for loads, not atomics. The atomic sectors
  per request in section 6 shows how spread out the atomics really are.

### iter06 (phase 1; the kernel the bullet is about)
1. **Packed argmax.** It clears the low 11 bits of each fp32 score and writes the centroid index there (2,048
   centroids need 11 bits). Clearing the bits rounds the score down a little, and the index changes it by less than
   that rounding step. So one `max` over the packed values compares the rounded scores and carries the winner's
   index in its low bits, in place of a max, an index search and a select.
   **The cost:** scores are compared with 12 fraction bits instead of 23. It can therefore pick a centroid whose
   score is below the best by up to 2^-12 of the score, which is at most 2^-12 × |key| because centroids have
   length 1. That bound is exactly phase 2's tolerance, which is 16× finer than bf16, the precision the keys are
   stored in. My CPU emulation of the packing gives a worst case of 1.9e-4 on these keys (gate layer 0), against the
   bound of 2.44e-4.
2. **Coalesced atomics.** After it stores the labels, it re-reads the keys (still in L2) as one flat list:
   `ii = tl.arange(0, EPI_ROWS * D)`, with row `ii // D` and column `ii % D`. Neighbouring threads therefore add
   neighbouring columns of one key into `s[label]`. In the agent's kernels the column offset is also multiplied by
   `stride_one`, a runtime 1 marked `do_not_specialize` so the compiler can't see that it is 1. The cell below shows
   whether iter06 does this. My guess is that it stops the compiler from handing each thread several neighbouring
   values, which suits loads but spreads out the atomics. The ablation in section 4 tests this guess.

### int8 (phase 1's fastest; fails on real keys)
- **Scoring:** keys and centroids are scored in int8 on tensor cores (on A100: 624 int8 TOPS vs 312 bf16 TFLOPS).
  Each key is scaled so its largest entry becomes 127, and all centroids share one scale.
- **Rescoring:** it keeps the 2 best approximate candidates per key, rescores those 2 exactly in fp32, and keeps
  the better one.
- **Failure:** it picks wrong whenever the true best centroid isn't among the 2 candidates. Its own tuner can also
  rescore 3 or 4, but 2 was fastest.
- **Where it fails:** on random keys the best few centroids score almost the same, so int8 rounding reorders them;
  my CPU emulation gives a cos gap of about 2e-3 with 2 candidates and 1e-3 even with 4. On real keys each key sits
  close to one k-means centroid, so the margins are wider. With 2 candidates it is exact on 13 of the 16 rows and
  fails on both layer-0 rows and layer 11 head 1 (cos gap up to 3e-3, 99.97% agreement); with 4 it passes all 16.
  `int8 x4` below is the same kernel limited to 4 candidates, to test that on the GPU. (On the old uncentered files
  int8 failed far worse: the bias made a few entries of each layer-0 key huge, so half the others rounded to 0.)

### phase2 (phase 2's best)
- **Persistent kernel:** it launches a fixed number of programs (about 3 per SM) that loop over the work.
- **Split tiles:** one tile's centroid loop can be split across programs. They merge their best (score, index)
  with a 64-bit `atomic_max`, and the last one to finish does the atomic adds.
- **Precision:** it packs only the 6-bit tile number into the score, keeping 17 fraction bits. Bound
  2^-17 ≈ 7.6e-6.
- **What it was tuned on:** the old uncentered gate keys, where the original kernel was very slow on the layer-0
  row. Its speedup there came from that row.

### Third pass (centered keys)
The agent loop was run again on the centered gate keys, starting from iter06 and told about phase 2's 6-bit packing.
In 6 iterations it found nothing that was both correct and faster than iter06, and stopped at its patience limit.
So iter06 is still the kernel to report.

The cell below prints the lines where these changes live; `!cat k_iter06.py` shows a whole file.

In [ ]:
import re
patterns = {"original": r"atomic_add|tl\.max\(|tl\.where",
            "iter06": r"atomic_add|MASK_HI|MASK_LO|stride_one|// D|% D|num_warps",
            "int8": r"atomic_add|tl\.int8|out_dtype|NCAND|127\.0",
            "phase2": r"atomic_add|atomic_max|MASK_HI|while it|maxnreg"}
for name, pat in patterns.items():
    print(f"===== k_{name}.py")
    for i, line in enumerate(open(f"k_{name}.py"), 1):
        if re.search(pat, line):
            print(f"{i:4d}  {line.rstrip()}")

## 3. Correctness

Each kernel's labels are compared with exact fp32 math on every key:
- **agree:** share of keys that get the same centroid as exact math. Near-ties can flip without harm.
- **abs gap:** worst (best score − chosen score).
- **cos gap:** the same gap divided by |key|. Centroids have length 1, so this is a gap in cosine similarity.
- **sums and counts:** `s` and `cnt` must match the kernel's own labels exactly.

Two rules:
- **Phase-1 rule** (phase 1 checked only synthetic keys): abs gap < 0.05.
- **Phase-2 rule:** cos gap ≤ 2^-12 ≈ 2.44e-4. It doesn't depend on key length. Centered real keys range from
  |key| ≈ 11 to ≈ 51 (layer 0 is longest), and the old uncentered layer-0 keys were ≈ 820, so a fixed absolute gap
  is strict on long keys and loose on short ones.

The agent's harness applies the phase-2 rule to synthetic keys as well as real ones.

In [ ]:
import statistics, lab
K = lab.kernels()
sets = {"synthetic": lab.synthetic(), "gate keys": lab.real(lab.GATE), "held-out keys": lab.real(lab.HELDOUT)}
label = lambda lh: f"L{lh[0]}h{lh[1]}" if lh[0] != "row" else f"row{lh[1]}"
for set_name, (keys, cent, rows) in sets.items():
    print(set_name)
    for name, launch in K.items():
        res = lab.check(keys, cent, *lab.run(launch, keys, cent))
        f1 = [label(lh) for lh, r in zip(rows, res) if not lab.rule1(r)]
        f2 = [label(lh) for lh, r in zip(rows, res) if not lab.rule2(r)]
        print(f"  {name:8s} phase-1 rule {'PASS' if not f1 else 'FAIL'} | phase-2 rule {'PASS' if not f2 else 'FAIL'} | "
              f"min agree {min(r['agree'] for r in res):.5f} | worst abs gap {max(r['abs_gap'] for r in res):.1e} | "
              f"worst cos gap {max(r['cos_gap'] for r in res):.1e} | failing rows: rule 1 {f1 or '-'}, rule 2 {f2 or '-'}")

What to look for (predictions from my CPU emulation):
- **int8** should pass the phase-1 rule on synthetic keys, which is how it got through phase 1, and fail the
  phase-2 rule there (cos gap ≈ 2e-3). So phase 1's loose absolute tolerance mattered as much as its synthetic
  data. On real keys it should fail both rules on the two layer-0 rows (abs gaps 0.06 and 0.13) and the phase-2
  rule on layer 11 head 1.
- **int8 x4** should pass the phase-2 rule on all real rows and still fail it on synthetic keys. The harness would
  reject it for the synthetic failure alone. Random keys are a harsher test than real ones here, because real keys
  sit near their centroids.
- **iter06** should pass the phase-2 rule everywhere, with a worst cos gap near 1.9e-4.
- On the old uncentered files, iter06 failed the phase-1 rule on layer 0 (abs gap 0.06, because |key| ≈ 820) even
  though its cos gap was only 7e-5. That mismatch is why phase 2 switched rules.

## 4. Speed

GPU clocks drift. In the first runs, timing two kernels in different processes and comparing the numbers was off by
up to ~18% (section 5 lets you test that). So here every kernel is timed in the same process, in alternating rounds,
and scored by the median per-round ratio to the original. Times are kernel-only: zeroing `s` and `cnt` is timed
separately and subtracted.

### Whole batches (8 rows at once)

In [ ]:
for set_name, (keys, cent, _) in sets.items():
    print(set_name)
    lab.show(lab.compare(K, keys, cent))

Earlier results: int8 1.74× on synthetic keys. On the old uncentered gate keys, phase2 was 2.36× and iter06 2.13×,
because the original was slow on the layer-0 row. Every row here is an ordinary head, so expect iter06 near 1.36×
and phase2 below it. `int8 x4` shows what the extra candidates cost.

### One (layer, head) at a time
Each row is copied into all 8 batch slots, so the shape (and each kernel's chosen config) stays the same. On the old
files' ordinary rows, iter06 was 1.35–1.38× and phase2 1.25–1.28×.

In [ ]:
pick = {k: K[k] for k in ["original", "iter06", "phase2"]}
table = []
for set_name in ["gate keys", "held-out keys"]:
    keys, cent, rows = sets[set_name]
    for i, (layer, head) in enumerate(rows):
        res = lab.compare(pick, *lab.one_row(keys, cent, i), rounds=5)
        table.append(res)
        print(f"L{layer:2d}h{head}  original {res['original'][0]:.3f} ms | iter06 {res['iter06'][0]:.3f} ms "
              f"{res['iter06'][1]:.2f}x | phase2 {res['phase2'][0]:.3f} ms {res['phase2'][1]:.2f}x")
print()
for k in ["iter06", "phase2"]:
    xs = [r[k][1] for r in table]
    print(f"{k}: {len(xs)} rows {min(xs):.2f}-{max(xs):.2f}x (median {statistics.median(xs):.2f}x)")

### Ablation: what each change buys
The five variants in `k_ablation.py` use the original's tile sizes, so only the two changes differ:

| Variant | Argmax | Atomic adds |
|---|---|---|
| 1 | as in the original | straight from the key tile, as in the original |
| 2 | packed | from the key tile |
| 3 | as in the original | flat re-read (iter06 style) |
| 4 | packed | flat re-read, with the compiler able to see the stride |
| 5 | packed | flat re-read, iter06 style |

Variant 1 should time close to the original. If it doesn't, the gap comes from details such as runtime sizes, so
compare the variants with each other. iter06 is included for reference; it may use different tile sizes.
Section 7 runs variant 5 with iter06's own launch settings.

In [ ]:
A = lab.ablation()
keys, cent, rows = sets["gate keys"]
print("correctness on gate keys (phase-2 rule)")
for name, launch in A.items():
    res = lab.check(keys, cent, *lab.run(launch, keys, cent))
    print(f"  {name:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")
for which in ["synthetic", "normal"]:
    print(f"\n{which}")
    lab.show(lab.compare({"original": K["original"], **A, "iter06": K["iter06"]}, *lab.inputs(which), rounds=7))

## 5. Harness checks

**Repeat-call check.** The timing loop calls a kernel hundreds of times, but only the first call is checked. A kernel
that skips the work on later calls would pass the check and time near zero. The harness therefore re-checks one call
made after timing. Below is a deliberately lazy wrapper around iter06.

In [ ]:
iter06, seen = K["iter06"], set()
def lazy(data, cent, s, cnt, m):
    """Does the work on the first call for each input, then returns immediately."""
    if data.data_ptr() in seen:
        return
    seen.add(data.data_ptr())
    iter06(data, cent, s, cnt, m)

keys, cent, _ = sets["synthetic"]
res = lab.check(keys, cent, *lab.run(lazy, keys, cent))
print("first call, checked:   ", "PASS" if all(map(lab.rule2, res)) else "FAIL")
t = lab.compare({"original": K["original"], "lazy": lazy}, keys, cent, rounds=3)
print(f"timed: lazy {t['lazy'][0] * 1e3:.0f} us per call vs original {t['original'][0] * 1e3:.0f} us")
res = lab.check(keys, cent, *lab.run(lazy, keys, cent))
print("one more call, checked:", "PASS" if all(map(lab.rule2, res)) else "FAIL  <- caught by the repeat-call check")

**Drift between processes.** Each run below is a fresh process timing the same original kernel, the way the first
agent runs timed each candidate in its own process. The spread is the error you'd get comparing numbers from
different processes. Drift depends on GPU temperature and boost state, so you may see less than the ~18% seen
earlier. If you do, don't quote 18% unless you can find it in the earlier logs.

In [ ]:
import subprocess
def one():
    r = subprocess.run(["python", "drift.py"], capture_output=True, text=True, cwd="/content")
    if r.returncode:
        raise RuntimeError(r.stderr[-1500:])
    return float(r.stdout.split()[-1])
ts = [one() for _ in range(6)]
print("original kernel, one fresh process each:", ", ".join(f"{t:.3f}" for t in ts), "ms")
print(f"spread: slowest / fastest = {max(ts) / min(ts):.3f}")

## 6. Nsight Compute: why each kernel is as fast as it is

`prof.py` makes one launch between `torch.cuda.profiler.start()` and `stop()`, and `ncu --profile-from-start off`
profiles only that launch. int8 launches three kernels (two small ones quantize the centroids); the table shows the
main one. Nsight replays each kernel many times and locks clocks at base frequency, so its durations are longer than
section 4's. Compare kernels with each other, not with section 4.

How to read the rows:
- **Compute / memory throughput %** ("speed of light"): how close the busiest compute unit and the busiest memory
  unit came to their peaks. If neither is near ~80%, hardware throughput isn't the limit; latency is, and warps sit
  waiting.
- **Tensor pipe %:** how busy the tensor cores are (bf16 for original, iter06 and phase2; int8 for int8).
- **DRAM %:** tiny for all of these. Keys and centroids are reused from cache, so memory bandwidth isn't the problem.
- **Registers, blocks/SM limits, achieved occupancy:** how many warps fit on an SM at once. More warps give the
  scheduler something to run while others wait. The original uses 166 registers and 41 KB of shared memory, and each
  of those alone limits it to 3 blocks (12 warps) per SM, which is 18.75% occupancy.
- **Warp cycles per issued instruction:** how long a warp waits, on average, between its instructions. The rows
  under it split that wait by reason:
  - *global loads* (long scoreboard): waiting for data from L2 or DRAM.
  - *global-memory queue full* (lg throttle): too many global-memory instructions in flight, typically atomics
    piling up.
  - *barrier:* waiting at `tl.debug_barrier()` for the rest of the block.
  - *tensor/math pipe busy:* the good kind of stall; the hardware is working.
  - *shared memory:* tensor-core operands pass through shared memory.
- **Atomic sectors per request:** 4 means perfectly coalesced (32 threads × 4 bytes = 128 bytes = 4 sectors). Higher
  means each atomic instruction is spread over more sectors. The count includes the `cnt` adds, which are a small
  share.

What to compare:
- original vs iter06 on `normal`: atomic sectors per request, lg throttle, duration.
- Each kernel on `synthetic` vs `normal`: real keys share centroids within a tile (9–23 distinct per 128 keys,
  against about 124 for random keys), so check whether the atomic stalls grow.
- int8 vs iter06: the int8 tensor pipe, and whether int8 still waits on the same things.

Takes about 5 minutes. Counters that this ncu version doesn't have are skipped and listed.

In [ ]:
import csv, subprocess
import pandas as pd

LABELS = {
    "gpu__time_duration.sum": "Duration (us, clocks locked)",
    "sm__throughput.avg.pct_of_peak_sustained_elapsed": "Compute (SM) throughput %",
    "gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed": "Memory throughput %",
    "l1tex__throughput.avg.pct_of_peak_sustained_active": "L1/TEX throughput %",
    "lts__throughput.avg.pct_of_peak_sustained_elapsed": "L2 throughput %",
    "dram__throughput.avg.pct_of_peak_sustained_elapsed": "DRAM throughput %",
    "sm__pipe_tensor_op_hmma_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, bf16 %",
    "sm__pipe_tensor_op_imma_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, int8 %",
    "sm__pipe_tensor_cycles_active.avg.pct_of_peak_sustained_active": "Tensor pipe, any type %",
    "launch__registers_per_thread": "Registers per thread",
    "launch__occupancy_limit_registers": "Blocks/SM allowed by registers",
    "launch__occupancy_limit_shared_mem": "Blocks/SM allowed by shared memory",
    "sm__warps_active.avg.pct_of_peak_sustained_active": "Achieved occupancy %",
    "smsp__average_warp_latency_per_inst_issued.ratio": "Warp cycles per issued instruction",
    "smsp__average_warps_issue_stalled_long_scoreboard_per_issue_active.ratio": "  global loads (long scoreboard)",
    "smsp__average_warps_issue_stalled_lg_throttle_per_issue_active.ratio": "  global-memory queue full (lg throttle)",
    "smsp__average_warps_issue_stalled_barrier_per_issue_active.ratio": "  barrier",
    "smsp__average_warps_issue_stalled_math_pipe_throttle_per_issue_active.ratio": "  tensor/math pipe busy",
    "smsp__average_warps_issue_stalled_mio_throttle_per_issue_active.ratio": "  shared-memory queue full (mio throttle)",
    "smsp__average_warps_issue_stalled_short_scoreboard_per_issue_active.ratio": "  shared-memory results (short scoreboard)",
    "smsp__average_warps_issue_stalled_wait_per_issue_active.ratio": "  fixed-latency wait",
    "l1tex__t_requests_pipe_lsu_mem_global_op_red.sum": "red requests",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_red.sum": "red sectors",
    "l1tex__t_requests_pipe_lsu_mem_global_op_atom.sum": "atom requests",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_atom.sum": "atom sectors",
}
SAFE = list(LABELS)[:6] + [m for m in LABELS if m.startswith(("launch__", "sm__warps_active"))]
known = {l.split()[0] for l in subprocess.run(["ncu", "--query-metrics"], capture_output=True, text=True).stdout.splitlines() if l.strip()}
metrics = [m for m in LABELS if m in SAFE or m.split(".")[0] in known]
print("skipped (not available):", [m for m in LABELS if m not in metrics] or "none")

def num(v):
    try:
        return float(v.replace(",", ""))
    except ValueError:
        return float("nan")

def profile(kernel, which, metrics):
    """One column of the table, or (None, ncu's output) if ncu failed."""
    cmd = ["ncu", "--profile-from-start", "off", "--csv", "--page", "raw", "--metrics", ",".join(metrics),
           "python", "prof.py", kernel, which]
    out = subprocess.run(cmd, capture_output=True, text=True, cwd="/content")
    lines = out.stdout.splitlines()
    start = next((i for i, l in enumerate(lines) if l.startswith('"ID"')), None)
    if start is None:
        return None, out.stdout[-1500:] + out.stderr[-1500:]
    rows = [r for r in csv.reader(lines[start:])]
    header = rows[0]
    units = next((dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and not r[0].strip()), {})
    launches = [dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and r[0].strip().isdigit()]
    main = max(launches, key=lambda d: num(d.get("gpu__time_duration.sum", "nan")))
    col = {LABELS[m]: num(main.get(m, "nan")) for m in metrics}
    scale = {"ns": 1e-3, "nsecond": 1e-3, "us": 1.0, "usecond": 1.0, "ms": 1e3, "msecond": 1e3}[units.get("gpu__time_duration.sum", "ns")]
    col["Duration (us, clocks locked)"] *= scale
    req = sum(col.get(k, 0) for k in ["red requests", "atom requests"])
    sec = sum(col.get(k, 0) for k in ["red sectors", "atom sectors"])
    for k in ["red requests", "atom requests", "red sectors", "atom sectors"]:
        col.pop(k, None)
    if req:
        col["Atomic requests (warp instructions)"] = req
        col["Atomic sectors per request"] = sec / req
    return col, None

cols = {}
for kernel in ["original", "iter06", "int8", "phase2"]:
    for which in ["synthetic", "normal"]:
        col, err = profile(kernel, which, metrics)
        if col is None and metrics != SAFE:
            print("ncu rejected a counter, retrying with the basic set. ncu said:\n" + err)
            metrics = SAFE
            col, err = profile(kernel, which, metrics)
        if col is None:
            raise RuntimeError(err)
        cols[f"{kernel}/{which}"] = col
        print("profiled", kernel, which)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
df = pd.DataFrame(cols)
fmt = lambda v: "-" if v != v else (f"{v:,.0f}" if abs(v) >= 100 else f"{v:.3g}")
print((df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string())

**Full Nsight report for one kernel.** Set `KERNEL` (original | iter06 | int8 | phase2, or a name from
`lab.ablation()`) and `INPUT` (synthetic | normal). The `OPT` / `INF` lines are Nsight's own suggestions.

In [ ]:
KERNEL, INPUT = "original", "normal"
!ncu --profile-from-start off --section SpeedOfLight --section ComputeWorkloadAnalysis --section MemoryWorkloadAnalysis --section Occupancy --section WarpStateStats --section LaunchStats python prof.py "{KERNEL}" {INPUT}

## 7. The rest of iter06's speedup, and where its fewer atomic sectors come from

Two questions the sections above leave open:
1. **The remaining ~9%.** With both changes at the original's launch settings, variant 5 reaches about 1.25×;
   iter06 reaches about 1.37×. The two are not launched the same way: in section 6's full reports, iter06 uses
   49,152 bytes of shared memory per block against the original's 40,960, and 168 registers against 166. Both still fit 3 blocks per SM, so occupancy doesn't change.
2. **The 3.9× fewer atomic sectors.** Both kernels issue the same number of atomic requests, but iter06's touch about
   3.9× fewer 32-byte sectors. That could come from the flat reload, the packed argmax or the hidden stride.

The first cell runs iter06 and variant 5 once each and prints every launch argument that isn't a tensor, plus what
the compiler used (warps, stages, shared memory, registers). It also prints iter06's kernel source, to compare with
`k_ablation.py`, and saves iter06's settings to `iter06_config.json`, which `lab.ablation()` turns into variants 6
and 7 below.

In [ ]:
import inspect, json
from triton.runtime.jit import JITFunction


def compiled(launch, keys, cent):
    """Run launch once and return the last Triton kernel it started: its source, its arguments by name, the compiled
    kernel (.asm holds the TTGIR, PTX and cubin; .metadata the warps, stages and shared memory) and the labels."""
    seen, run = [], JITFunction.run

    def rec(self, *args, **kwargs):
        kernel = run(self, *args, **kwargs)
        seen.append((self, {**dict(zip(getattr(self, "arg_names", []), args)), **kwargs}, kernel))
        return kernel

    JITFunction.run = rec
    try:
        _, _, m = lab.run(launch, keys, cent)
    finally:
        JITFunction.run = run
    fn, params, kernel = seen[-1]  # iter06 tunes on its first call; the last launch is the config it chose
    return {"src": inspect.getsource(fn.fn), "params": params, "kernel": kernel, "labels": m}


def meta(kernel, name):
    m = kernel.metadata
    return m.get(name) if isinstance(m, dict) else getattr(m, name, None)


def settings(e):
    """Every launch argument that isn't a tensor, and what the compiler used."""
    out = {k: v for k, v in e["params"].items() if isinstance(v, (bool, int, float, str)) and k != "warmup"}
    out["grid"] = e["params"].get("grid")
    out.update(num_warps=meta(e["kernel"], "num_warps"), num_stages=meta(e["kernel"], "num_stages"),
               shared_bytes=meta(e["kernel"], "shared"), registers=getattr(e["kernel"], "n_regs", None))
    return out


def tags():
    """Short names used in the tables and in ir/: original, v1-v5, v6 and v7 (iter06's settings; v7 only if iter06
    reduces each tile), v8 and v9 (tile atomics with the hidden stride), v10 (deferred argmax without packing),
    v11 and v12 (v4 with the stride hidden from the atomics or the loads only), v4r168 and v5r128 (register caps),
    iter06."""
    fixed = {**{n: f"v{i}" for i, n in enumerate(lab.ABLATION, 1)}, "v5 + iter06 settings": "v6",
             "v5 + iter06 settings + reduce": "v7", **{n: f"v{i}" for i, n in enumerate(lab.TILE_STRIDE, 8)},
             **{n: f"v{i}" for i, n in enumerate(lab.DEFERRED, 10)},
             **{n: f"v{i}" for i, n in enumerate(lab.STRIDE_SPLIT, 11)},
             **{n: f"v{4 if epi == 1 else 5}r{r}" for n, (_, epi, r) in lab.REG_CAP.items()}}
    return {"original": "original", **{n: fixed[n] for n in lab.ablation()}, "iter06": "iter06"}


keys, cent = lab.inputs("normal")
e5 = compiled(lab.ablation()["packed, flat (iter06 style)"], keys, cent)
e06 = compiled(K["iter06"], keys, cent)
s5, s06 = settings(e5), settings(e06)
print(f"{'':16s} {'variant 5':>22s} {'iter06':>22s}")
for name in [*s5, *(k for k in s06 if k not in s5)]:
    a, b = s5.get(name, "-"), s06.get(name, "-")
    print(f"{name:16s} {str(a):>22s} {str(b):>22s}{'   <- differs' if a != b else ''}")

print("\niter06's kernel source:\n")
print(e06["src"])

defaults = {"BLOCK_N": 128, "BLOCK_K": 32, "EPI_ROWS": 32, "REDUCE_EACH": False}
missing = [k for k in defaults if k not in s06]
if missing:
    print(f"iter06 has no argument named {missing}; variants 6 and 7 keep variant 5's value for those")
cfg = {"block_n": s06.get("BLOCK_N", 128), "block_k": s06.get("BLOCK_K", 32), "epi_rows": s06.get("EPI_ROWS", 32),
       "reduce_each": bool(s06.get("REDUCE_EACH", False)),
       "num_warps": s06["num_warps"], "num_stages": s06["num_stages"]}
(lab.HERE / "iter06_config.json").write_text(json.dumps(cfg))
print("variants 6 and 7 use:", cfg)

**Variants 6 and 7.** Variant 6 is variant 5 run with iter06's tile sizes, warps, stages and epilogue rows. If
iter06 reduces each tile inside the loop (`REDUCE_EACH`), variant 7 adds that as well. The cell checks both on the gate
keys, then times them against variant 5 and iter06 on `normal`. How to read the timing:
- **Variant 6 close to iter06:** the rest of the gap is launch settings; the table above shows which ones differ.
- **Only variant 7 close:** iter06's per-tile reduction matters as well.
- **Neither close:** a code difference remains. Compare iter06's source above with `k_ablation.py`.

In [ ]:
A = lab.ablation()  # reads iter06_config.json, so it now includes variants 6 and 7
new = {name: launch for name, launch in A.items() if "iter06 settings" in name}
keys, cent, rows = sets["gate keys"]
print("correctness on gate keys (phase-2 rule)")
for name, launch in new.items():
    res = lab.check(keys, cent, *lab.run(launch, keys, cent))
    print(f"  {name:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")
print("\nnormal")
pick = {"original": K["original"], "packed, flat (iter06 style)": A["packed, flat (iter06 style)"], **new,
        "iter06": K["iter06"]}
lab.show(lab.compare(pick, *lab.inputs("normal"), rounds=7))

### Which change cuts the atomic sectors

Each variant switches on one change at a time, so comparing their atomic counters isolates each change:

| | Atomics from the key tile | Flat reload, stride visible | Flat reload, iter06 style |
|---|---|---|---|
| plain argmax | v1 | | v3 |
| packed argmax | v2 | v4 | v5 |

- **Packing:** v1 → v2 and v3 → v5.
- **Flat reload:** v1 → v3 and v2 → v5.
- **Hidden stride:** v4 → v5.

My prediction before the first run: packing leaves sectors per request unchanged, because it changes how the label is
found but not which addresses the atomics touch. Flat reload iter06-style takes them from about 16 to about 4, and v4
stays near 16. If so, the drop comes from how the threads are assigned to the flat reload's elements, which the cell
after this one reads from the compiled code.

The cell below profiles the original, every variant and iter06 on `normal`. It reuses section 6's `profile()` (run
section 6's table cell first) and adds executed warp instructions, so the table also shows where the instruction
count drops. It takes about 5 minutes.

In [ ]:
LABELS["smsp__inst_executed.sum"] = "Executed warp instructions"
keep = ["gpu__time_duration.sum", "launch__registers_per_thread", "launch__occupancy_limit_registers",
        "launch__occupancy_limit_shared_mem", "sm__warps_active.avg.pct_of_peak_sustained_active",
        "smsp__average_warps_issue_stalled_lg_throttle_per_issue_active.ratio",
        "l1tex__t_requests_pipe_lsu_mem_global_op_red.sum", "l1tex__t_sectors_pipe_lsu_mem_global_op_red.sum",
        "l1tex__t_requests_pipe_lsu_mem_global_op_atom.sum", "l1tex__t_sectors_pipe_lsu_mem_global_op_atom.sum"]
ab_metrics = [m for m in keep if m in metrics] + [m for m in ["smsp__inst_executed.sum"] if m.split(".")[0] in known]
T = tags()
ncu_ab = {}
for name, tag in T.items():
    col, err = profile(name, "normal", ab_metrics)
    if col is None:
        raise RuntimeError(err)
    ncu_ab[name] = col
    print("profiled", tag, name)
df = pd.DataFrame({T[name]: col for name, col in ncu_ab.items()})
print((df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string())

### How the threads are assigned

Triton decides which thread holds which element of every tensor. The decision is recorded as a **layout** in the
TTGIR (Triton GPU IR, the last stage before LLVM and PTX), for example
`#blocked = #ttg.blocked<{sizePerThread = [4], threadsPerWarp = [32], warpsPerCTA = [4], order = [0]}>`:
- **sizePerThread:** how many neighbouring elements one thread holds, per dimension.
- **threadsPerWarp / warpsPerCTA:** how the 32 lanes of a warp, and the warps of a block, spread over the dimensions.
- **order:** which dimension is fastest, i.e. whose neighbouring elements go to neighbouring lanes.

Each atomic **request** in Nsight is one warp instruction: the 32 lanes each add the element they hold in the same
register slot. A100 has no vector fp32 atomic add (PTX adds `.v2`/`.v4.f32` reductions only from sm_90), so every
element is its own instruction. That is why the request count is the same for every variant: each thread issues one
atomic per element it holds, 128 for `s` plus 1 for `cnt` (512 + 4 requests per block). The layout decides which
addresses those 32 lanes hit. With `sizePerThread = [1]`, lane i holds column i: one request covers 128 consecutive
bytes, 4 sectors. With `sizePerThread = [4]`, lane i holds columns 4i to 4i+3, so one request covers every fourth
column across 512 bytes, 16 sectors.

The cell below reads each kernel's layouts from its TTGIR and prints, for one atomic instruction of warp 0, which key
and column each lane adds. It then replays every atomic of every block with the labels the kernel actually produced
and counts the 32-byte sectors per request that this layout implies, two ways: distinct sectors, and the
same count when lanes that hit the same address are serviced in separate passes (a sector then counts once per pass,
as many times as its most-hit address). The second is a counting model, not documented hardware behavior. If the prediction matches Nsight's count above
for every variant, the layout explains the sectors. PTX shows the same thing only indirectly, as per-thread address
arithmetic; the layout says it directly.

In [ ]:
import math, re
import torch

BLOCKED = re.compile(r"^#(\w+) = #(?:ttg|triton_gpu)\.blocked<\{sizePerThread = \[([\d, ]*)\], threadsPerWarp = "
                     r"\[([\d, ]*)\], warpsPerCTA = \[([\d, ]*)\], order = \[([\d, ]*)\]", re.M)
ATOMIC = re.compile(r"tt\.atomic_rmw.*?tensor<([\dx]+)x!tt\.ptr<(\w+)>, (#[\w.]+(?:<\{[^}]*\}>)?)>")
SLICE = re.compile(r"#(?:ttg|triton_gpu)\.slice<\{dim = (\d+), parent = #(\w+)\}>")
ints = lambda s: [int(v) for v in s.replace("x", ",").split(",")]


def owners(shape, spt, tpw, wpc, order):
    """Where a blocked layout puts each element (row-major index): its coordinates, warp, register slot and lane."""
    n = math.prod(shape)
    rest, coords = torch.arange(n), [None] * len(shape)
    for d in reversed(range(len(shape))):
        coords[d], rest = rest % shape[d], rest // shape[d]
    warp, slot, lane = (torch.zeros(n, dtype=torch.long) for _ in range(3))
    lm = wm = sm = 1
    for d in order:  # order[0] is the dimension whose neighbouring elements go to neighbouring lanes
        lane += coords[d] // spt[d] % tpw[d] * lm
        warp += coords[d] // (spt[d] * tpw[d]) % wpc[d] * wm
        slot += coords[d] % spt[d] * sm
        lm, wm, sm = lm * tpw[d], wm * wpc[d], sm * spt[d]
    for d in order:  # a tensor bigger than one pass of the layout wraps around into more register slots
        tile = spt[d] * tpw[d] * wpc[d]
        slot += coords[d] // tile * sm
        sm *= -(-shape[d] // tile)
    return coords, warp, slot, lane


def atomics(kernel):
    """Each atomic in the TTGIR: (shape, element type, layout text, decoded layout or None, sliced-away dim or None)."""
    ttgir = kernel.asm["ttgir"]
    blocked = {m[0]: [ints(x) for x in m[1:]] for m in BLOCKED.findall(ttgir)}
    out = []
    for shape, elem, text in ATOMIC.findall(ttgir):
        sl = SLICE.fullmatch(text)
        if text[1:] in blocked:
            out.append((ints(shape), elem, text, blocked[text[1:]], None))
        elif sl and sl[2] in blocked:  # a slice drops one dimension of its parent layout
            out.append((ints(shape), elem, text, blocked[sl[2]], int(sl[1])))
        else:
            out.append((ints(shape), elem, text, None, None))
    return out


def steps(xs):
    if len(xs) > 2 and len({b - a for a, b in zip(xs, xs[1:])}) == 1:
        return f"{xs[0]}, {xs[1]}, ..., {xs[-1]} (step {xs[1] - xs[0]})"
    return str(xs)


def replay(e, d=128):
    """Predicted requests per block and sectors per request of each atomic, from the TTGIR layouts and the labels the
    kernel produced. Uses batch slot 0; on `normal` all 8 slots hold the same row."""
    bn = e["params"].get("BLOCK_N", 128)
    labels = e["labels"][0].long().view(-1, bn)  # [blocks, keys]
    dev, nblk = labels.device, labels.shape[0]
    n_words = ((int(labels.max()) + 1) * d + 7) // 8 * 8  # address range per request, a whole number of sectors
    total_req = total_sec = total_pass = 0
    for shape, elem, text, layout, drop in atomics(e["kernel"]):
        what = "cnt" if elem == "i32" else "s"
        if layout is None:
            print(f"  {what:3s} {shape} {elem}: layout {text} not decoded")
            continue
        print(f"  {what:3s} {shape} {elem}: sizePerThread {layout[0]}, threadsPerWarp {layout[1]}, "
              f"warpsPerCTA {layout[2]}, order {layout[3]}{f' (slice of dim {drop})' if drop is not None else ''}")
        pshape = shape if drop is None else shape[:drop] + [1] + shape[drop:]
        coords, warp, slot, lane = (t.to(dev) if torch.is_tensor(t) else [c.to(dev) for c in t]
                                    for t in owners(pshape, *layout))
        if drop is not None:
            coords = coords[:drop] + coords[drop + 1:]
        if what == "cnt":  # cnt[label] += 1, one element per key
            key, col, per_iter = coords[0], torch.zeros_like(coords[0]), bn
        elif len(shape) == 2:  # s[label, :] += key, the whole [keys, dims] tile at once
            key, col, per_iter = coords[0], coords[1], bn
        else:  # flat reload, EPI_ROWS keys per loop iteration: element i is key i // d, column i % d
            key, col, per_iter = coords[0] // d, coords[0] % d, shape[0] // d
        one = ((warp == 0) & (slot == 0)).nonzero().flatten()
        one = one[lane[one].argsort()]
        print(f"      warp 0, one instruction: lanes 0-{len(one) - 1} add keys {steps(sorted(set(key[one].tolist())))}"
              + ("" if what == "cnt" else f", columns {steps(col[one].tolist())}"))
        n_slot = int(slot.max()) + 1
        group = warp * n_slot + slot  # one group = one warp instruction
        n_group, g_max = int(torch.unique(group).numel()), (int(warp.max()) + 1) * n_slot
        sec = passes = 0
        for r0 in range(0, bn, per_iter):
            word = labels[:, r0 + key] * (1 if what == "cnt" else d) + col  # [blocks, elements], 4-byte words
            addr, hits = torch.unique((torch.arange(nblk, device=dev)[:, None] * g_max + group) * n_words + word,
                                      return_counts=True)  # each address once per request, with how many lanes hit it
            sector, inv = torch.unique(addr // 8, return_inverse=True)
            sec += sector.numel()
            # if lanes that hit the same address are serviced in separate passes, a sector is touched once per pass,
            # i.e. as many times as its most-hit address
            passes += int(torch.zeros_like(sector).scatter_reduce(0, inv, hits, "amax").sum())
        req = n_group * (bn // per_iter)
        print(f"      {req} requests per block, {sec / (req * nblk):.2f} distinct sectors per request, "
              f"{passes / (req * nblk):.2f} with same-address lanes in separate passes")
        total_req, total_sec, total_pass = total_req + req, total_sec + sec, total_pass + passes
    n = total_req * nblk
    return total_req, (total_sec / n, total_pass / n) if total_req else (float("nan"), float("nan"))


def report(tag, name, e):
    """Layouts, one warp instruction per atomic, and predicted (vs Nsight, if profiled) sectors per request."""
    print(f"{tag}  {name}")
    req, (spr, spr_pass) = replay(e)
    col = globals().get("ncu_ab", {}).get(name, {})
    nblk = e["labels"].numel() // e["params"].get("BLOCK_N", 128)
    measured = (f"; Nsight: {col['Atomic requests (warp instructions)'] / nblk:.0f} requests per block, "
                f"{col['Atomic sectors per request']:.2f} sectors per request" if "Atomic sectors per request" in col else "")
    print(f"  all atomics, predicted: {req} requests per block, {spr:.2f} distinct sectors per request, "
          f"{spr_pass:.2f} with same-address passes{measured}\n")


keys, cent = lab.inputs("normal")
T = tags()
comp = {name: compiled(launch, keys, cent) for name, launch in {"original": K["original"], **lab.ablation(),
                                                                 "iter06": K["iter06"]}.items()}
for name, e in comp.items():
    report(T[name], name, e)

### TTGIR, PTX and SASS

The cell below saves each kernel's TTGIR, PTX and SASS to `MyDrive/kernel-agent/ir/<name>/` so they can go into the
repo, and counts SASS instructions by opcode.
- **TTGIR** shows the layouts above and the software pipeline (how many centroid tiles are in flight).
- **PTX** is NVIDIA's virtual instruction set. `ptxas` still allocates registers and schedules instructions after it,
  so PTX instruction counts aren't final.
- **SASS** is the machine code the GPU runs. Its counts are static: a loop body counts once, however many times it
  runs. Executed counts are in the Nsight table above.

Opcodes counted: `HMMA` tensor-core multiply, `LDGSTS` global-to-shared copy (the centroid pipeline), `LDSM`/`LDS`/`STS`
shared-memory loads and stores, `BAR` barrier, `SHFL` warp shuffle (reductions across lanes), `FMNMX` float max,
`RED`/`ATOMG` global atomics without and with a returned value, `LDG`/`STG` global loads and stores.

In [ ]:
import collections, os, shutil, subprocess, tempfile
import triton.backends.nvidia

CUOBJDUMP = shutil.which("cuobjdump") or os.path.join(os.path.dirname(triton.backends.nvidia.__file__), "bin", "cuobjdump")
OPS = ["HMMA", "LDGSTS", "LDSM", "LDS", "STS", "BAR", "SHFL", "FMNMX", "RED", "ATOMG", "LDG", "STG"]
OUT = f"{D}/ir" if triton.__version__ == "3.6.0" else f"{D}/ir-triton-{triton.__version__}"  # repo ir/ = 3.6.0
SASS_HEADER = f"{'':9s}{'stages':>7s}{'shared':>8s}{'regs':>5s}{'total':>7s}" + "".join(f"{op:>7s}" for op in OPS)


def save_ir(tag, k):
    """Write the kernel's TTGIR, PTX and SASS to OUT/<tag>/ and print its static SASS instruction counts."""
    with tempfile.NamedTemporaryFile(suffix=".cubin") as f:
        f.write(k.asm["cubin"])
        f.flush()
        sass = subprocess.run([CUOBJDUMP, "-sass", f.name], capture_output=True, text=True, check=True).stdout
    os.makedirs(f"{OUT}/{tag}", exist_ok=True)
    for ext, text in [("ttgir", k.asm["ttgir"]), ("ptx", k.asm["ptx"]), ("sass", sass)]:
        with open(f"{OUT}/{tag}/{tag}.{ext}", "w") as f:
            f.write(text)
    ops = collections.Counter(re.findall(r"/\*[0-9a-f]{4,}\*/\s+(?:@!?U?P\w+\s+)?([A-Z0-9]+)", sass))
    print(f"{tag:9s}{meta(k, 'num_stages')!s:>7s}{meta(k, 'shared')!s:>8s}{getattr(k, 'n_regs', '-')!s:>5s}{sum(ops.values()):7d}"
          + "".join(f"{ops[op]:7d}" for op in OPS))


T = tags()
print(f"static SASS instruction counts; files in {OUT}/<name>/\n")
print(SASS_HEADER)
for name, e in comp.items():
    save_ir(T[name], e["kernel"])

### Runtime stride on the tile atomics (v8, v9)

v4 and v5 show that the flat reload alone isn't enough: the hidden stride is what changes the layout. That leaves one
question: does hiding the stride fix the original's tile atomics without the flat reload? v8 and v9 are v1 and v2
with the tile atomic's column offset multiplied by `stride_one` (`EPI = 3` in `k_ablation.py`). Nothing else
changes: no label store and reload, no key re-read.

The answer isn't obvious. For a [keys × columns] tile the compiler also chooses which dimension the 32 lanes of a
warp run along. Lanes along columns would give about 4 sectors per request, as in v5. Lanes along keys would put every
lane on a different key's row, which could be worse than 16. The tile's keys also still have to be converted from the
layout they were loaded in to the atomic's layout.

How to read it (compare v8 with v1 and v3, and v9 with v2 and v5):
- **About 4 sectors and as fast as the flat reload:** the hidden stride alone fixes the tile atomics; the reload
  isn't needed for locality.
- **About 4 sectors but slower than the flat reload:** locality is fixed, and the rest is other work, such as a
  `ttg.convert_layout` before the atomic in `ir/v8/v8.ttgir`.
- **16 sectors or more:** in 2D the compiler puts the lanes elsewhere, and the flat 1D reload is what gives each warp
  instruction consecutive columns.

The cell reuses section 6's `profile()` and the section 7 cells above, so run those first in the same session. It
takes about 3 minutes.

In [ ]:
A, T = lab.ablation(), tags()
new = {name: A[name] for name in lab.TILE_STRIDE}
keys, cent, rows = sets["gate keys"]
print("correctness on gate keys (phase-2 rule)")
for name, launch in new.items():
    res = lab.check(keys, cent, *lab.run(launch, keys, cent))
    print(f"  {name:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")

order = ["original", "plain argmax, tile atomics", "plain, tile, runtime stride", "plain argmax, flat atomics",
         "packed argmax, tile atomics", "packed, tile, runtime stride", "packed, flat (iter06 style)"]
print("\nnormal")
lab.show(lab.compare({"original": K["original"], **{n: A[n] for n in order[1:]}}, *lab.inputs("normal"), rounds=7))

for name in new:  # same counters as the Nsight table above
    col, err = profile(name, "normal", ab_metrics)
    if col is None:
        raise RuntimeError(err)
    ncu_ab[name] = col
df = pd.DataFrame({T[n]: ncu_ab[n] for n in order if n in ncu_ab})
print("\n" + (df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string() + "\n")

keys, cent = lab.inputs("normal")
for name, launch in new.items():
    comp[name] = compiled(launch, keys, cent)
    report(T[name], name, comp[name])
print(SASS_HEADER)
for name in new:
    save_ir(T[name], comp[name]["kernel"])

### Deferred argmax without packing (v10)

iter06's argmax change does two things at once: it defers the per-tile reduction to after the loop, and packing lets
one `max` carry the index along. v10 separates them. It defers the reduction the same way, but keeps two [keys ×
centroids] tiles, one of best scores and one of their centroid indices. Each iteration compares the new scores with
the best (`ip > best`), selects the index where they're higher (`tl.where`), and keeps the higher score. After the
loop it reduces once: each key's best score, then the lowest index among the columns holding it. No score bits are
lost, so it should pick exactly what v3 picks.

Compare v10 with v3 (same epilogue, argmax reduced in every tile) and v5 (same epilogue, packed):
- **Executed instructions near v5's:** deferring the reduction is what saves the instructions; packing adds little.
- **Near v3's:** the per-element compare and selects cost about what the per-tile reduction did, and packing is what
  makes deferral pay.
- **Registers and occupancy:** the index tile costs 32 more registers per thread at these tile sizes (4,096 indices
  over 128 threads). If occupancy drops below v3's and v5's 18%, the time comparison mixes the two effects.

Takes about 3 minutes. Rerun the `%%writefile k_ablation.py` and `lab.py` cells and section 7's first cell (it
defines `tags()`), then this cell. It reuses section 6's `profile()` and section 7's Nsight and SASS cells, and only
profiles the variants this session hasn't.

In [ ]:
import importlib
lab = importlib.reload(lab)  # picks up the rewritten lab.py without restarting the session
A, T = lab.ablation(), tags()
name = next(iter(lab.DEFERRED))
keys, cent, rows = sets["gate keys"]
res = lab.check(keys, cent, *lab.run(A[name], keys, cent))
print("correctness on gate keys (phase-2 rule)")
print(f"  {name:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")

order = ["plain argmax, flat atomics", name, "packed, flat (iter06 style)"]
print("\nnormal")
lab.show(lab.compare({"original": K["original"], **{n: A[n] for n in order}}, *lab.inputs("normal"), rounds=7))

for n in order:  # same counters as the Nsight table above
    if n not in ncu_ab:
        col, err = profile(n, "normal", ab_metrics)
        if col is None:
            raise RuntimeError(err)
        ncu_ab[n] = col
df = pd.DataFrame({T[n]: ncu_ab[n] for n in order})
print("\n" + (df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string() + "\n")

keys, cent = lab.inputs("normal")
print(SASS_HEADER)
for n in order:
    if n not in comp:
        comp[n] = compiled(A[n], keys, cent)
    save_ir(T[n], comp[n]["kernel"])

### Why v4 is slower than v5

v4 and v5 differ in one line: v5 multiplies the epilogue's column offset by `stride_one`. On Triton 3.6 that also
changed the atomic sectors (16.03 vs 4.12), and the sections above read v5's speed as coming from that. On Triton 3.8,
PR #10059 gives both 4.12 sectors, yet v4 still takes 1,037 µs to v5's 838 µs. So hiding the stride changes more than
the atomics' sectors. From the TTGIR, SASS and Nsight so far, it changes:
- **Atomic sectors:** 16.03 vs 4.12 on 3.6; 4.12 for both on 3.8.
- **A layout conversion:** v4 reloads keys 8 per thread (one 16-byte load), but its atomic takes 4 per thread on 3.6
  and 1 on 3.8, so `ttg.convert_layout` passes the keys through shared memory, with barriers. v5 loads and adds with
  the same layout and has no conversion.
- **Load width:** v4 reloads keys with 16-byte loads; v5 with one 2-byte load per element.
- **Registers, so blocks per SM:** on 3.6, v4 has 128 registers (4 blocks per SM, 23.6% occupancy) against v5's 130
  (3 blocks, 18%); on 3.8, v4 has 208 (2 blocks, 12.3%).

Triton's coalesce pass couples the load and the atomic. A load takes the widest per-thread width among the memory
accesses connected to it (the reloaded keys feed the atomic), and an atomic takes the smaller of that and its own
width. So hiding the stride from one side only separates the factors:

| | Atomic sees the contiguity | Atomic doesn't |
|---|---|---|
| **Load sees the contiguity** | v4 | v11 |
| **Load doesn't** | v12 | v5 |

Predictions from the coalesce pass (check the layout lines the cell prints):
- **v11:** 16-byte loads, one element per thread in the atomic, a conversion between them, 4.12 sectors. v11 vs v5
  isolates the conversion and load width at equal sectors; v4 vs v11 isolates the atomic sectors (3.6 only).
- **v12:** on 3.6, the load takes the atomic's width, so both use 4 per thread with no conversion and 16.03 sectors;
  v12 vs v5 isolates the sectors without a conversion. On 3.8 v12 should compile like v5.
- **Register caps** (`maxnreg`, passed to ptxas): v4 at 168 registers or fewer fits 3 blocks per SM like v5 on 3.8;
  v5 at 128 or fewer fits 4 like v4 on 3.6. If a cap closes the gap, blocks per SM explain it. A cap that forces
  spills adds local-memory traffic, so check the spills column.

What the extra Nsight rows show:
- **Occupancy** (block limits, theoretical, achieved) and **active and eligible warps per scheduler:** how many warps
  each scheduler can choose from, and how many are ready to issue. Under 1 eligible warp per cycle means a scheduler
  often has nothing to issue.
- **Stall reasons:** where waiting warps wait. A conversion should show as barrier and shared-memory (short
  scoreboard, mio throttle) stalls; atomic traffic as lg throttle and drain; fewer warps per SM as fewer eligible
  warps with the same stall mix.
- **Shared-memory instructions and bank conflicts:** the conversion's cost.
- **Global load requests and sectors per request:** v4's 16-byte reloads against v5's 2-byte ones.

**Coarsening.** The cell also times v4 and v5 at `EPI_ROWS` 8, 16, 32 and 64, the number of elements each thread loads
and adds per epilogue chunk. More elements per thread give each warp more independent loads and atomics to overlap
(instruction-level parallelism), at the cost of registers; fewer warps per SM leave more of the latency to that
overlap. If v4's gap shrinks with fewer, larger chunks, its conversion costs about the same per chunk.

Takes about 10 minutes. Rerun the `%%writefile k_ablation.py` and `lab.py` cells and section 7's first cell (it
defines `tags()`), then this cell. It reuses section 6's `profile()` and section 7's layout and SASS cells.

In [ ]:
import importlib, re
import k_ablation
lab = importlib.reload(lab)  # picks up the rewritten lab.py and k_ablation.py without restarting the session
k_ablation = importlib.reload(k_ablation)
A, T = lab.ablation(), tags()
V4, V5 = "packed, flat (stride visible)", "packed, flat (iter06 style)"
DIVE = [V4, *lab.STRIDE_SPLIT, V5, *lab.REG_CAP]

keys, cent, rows = sets["gate keys"]
print("correctness on gate keys (phase-2 rule)")
for n in DIVE:
    res = lab.check(keys, cent, *lab.run(A[n], keys, cent))
    print(f"  {T[n]:7s} {n:32s} {'PASS' if all(map(lab.rule2, res)) else 'FAIL'}  worst cos gap {max(r['cos_gap'] for r in res):.1e}")

print("\nnormal")
lab.show(lab.compare({"original": K["original"], **{n: A[n] for n in DIVE}}, *lab.inputs("normal"), rounds=7))

print("\ncoarsening: EPI_ROWS = elements each thread loads and adds per epilogue chunk (normal)")
sweep = {f"{tag}, EPI_ROWS {r}": k_ablation.variant(True, epi, epi_rows=r) for tag, epi in [("v4", 1), ("v5", 2)]
         for r in (8, 16, 32, 64)}
lab.show(lab.compare({"original": K["original"], **sweep}, *lab.inputs("normal"), rounds=5))

DIVE_LABELS = {
    "smsp__inst_executed.sum": "Executed warp instructions",
    "launch__block_size": "Threads per block",
    "launch__occupancy_limit_warps": "Blocks/SM allowed by warps",
    "sm__maximum_warps_per_active_cycle_pct": "Theoretical occupancy %",
    "smsp__warps_active.avg.per_cycle_active": "Active warps per scheduler",
    "smsp__warps_eligible.avg.per_cycle_active": "Eligible warps per scheduler",
    "smsp__issue_active.avg.pct_of_peak_sustained_active": "Cycles with an instruction issued %",
    "smsp__average_warps_issue_stalled_not_selected_per_issue_active.ratio": "  not selected (ready, another warp issued)",
    "smsp__average_warps_issue_stalled_drain_per_issue_active.ratio": "  drain (stores and atomics at exit)",
    "smsp__average_warps_issue_stalled_membar_per_issue_active.ratio": "  memory barrier",
    "smsp__average_warps_issue_stalled_dispatch_stall_per_issue_active.ratio": "  dispatch stall",
    "smsp__average_warps_issue_stalled_no_instruction_per_issue_active.ratio": "  no instruction",
    "smsp__average_warps_issue_stalled_branch_resolving_per_issue_active.ratio": "  branch resolving",
    "l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum": "global load requests",
    "l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum": "global load sectors",
    "smsp__inst_executed_op_shared_ld.sum": "Shared-memory load instructions",
    "smsp__inst_executed_op_shared_st.sum": "Shared-memory store instructions",
    "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum": "Shared-memory load bank conflicts",
    "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum": "Shared-memory store bank conflicts",
}
LABELS.update(DIVE_LABELS)
first = ["gpu__time_duration.sum", "launch__block_size", "launch__registers_per_thread",
         "launch__occupancy_limit_registers", "launch__occupancy_limit_shared_mem", "launch__occupancy_limit_warps",
         "sm__maximum_warps_per_active_cycle_pct", "sm__warps_active.avg.pct_of_peak_sustained_active",
         "smsp__warps_active.avg.per_cycle_active", "smsp__warps_eligible.avg.per_cycle_active",
         "smsp__issue_active.avg.pct_of_peak_sustained_active", "smsp__inst_executed.sum"]
memory = ["l1tex__t_requests_pipe_lsu_mem_global_op_red.sum", "l1tex__t_sectors_pipe_lsu_mem_global_op_red.sum",
          "l1tex__t_requests_pipe_lsu_mem_global_op_atom.sum", "l1tex__t_sectors_pipe_lsu_mem_global_op_atom.sum",
          *[m for m in DIVE_LABELS if m.startswith(("l1tex__", "smsp__inst_executed_op"))]]
wanted = list(dict.fromkeys(first + [m for m in LABELS if m.startswith("smsp__average_")] + memory))
ok = lambda m: m.startswith("launch__") or m in SAFE or m.split(".")[0] in known
dive_metrics = [m for m in wanted if ok(m)]
print("\nskipped (not available):", [m for m in wanted if not ok(m)] or "none")

ncu_dive = {}
for n in DIVE:
    col, err = profile(n, "normal", dive_metrics)
    if col is None:
        raise RuntimeError(err)
    req, sec = col.pop("global load requests", 0), col.pop("global load sectors", 0)
    if req:
        col["Global load requests"], col["Global load sectors per request"] = req, sec / req
    ncu_dive[n] = col
    print("profiled", T[n], n)
df = pd.DataFrame({T[n]: c for n, c in ncu_dive.items()})
print("\n" + (df.map(fmt) if hasattr(df, "map") else df.applymap(fmt)).to_string() + "\n")

RELOAD = re.compile(r"tt\.load .*?: tensor<(\d+)x!tt\.ptr<bf16>, #(\w+)>")  # 1-D key reloads
CONVERT = re.compile(r"ttg\.convert_layout .*?: tensor<\d+x(?!\d)")  # 1-D layout conversions
keys, cent = lab.inputs("normal")
print("epilogue layouts from the TTGIR: elements per thread in the key reload and the sum atomic")
for n in DIVE:
    if n not in comp:
        comp[n] = compiled(A[n], keys, cent)
    k = comp[n]["kernel"]
    ttgir = k.asm["ttgir"]
    blocked = {m[0]: [ints(x) for x in m[1:]] for m in BLOCKED.findall(ttgir)}
    reload = sorted({blocked[lay][0][0] for _, lay in RELOAD.findall(ttgir) if lay in blocked})
    add = [a[3][0][0] for a in atomics(k) if a[1] == "f32" and a[3]]
    print(f"  {T[n]:7s} reload {reload}  atomic {add}  1-D layout conversions {len(CONVERT.findall(ttgir))}  "
          f"registers {k.n_regs}  spills {getattr(k, 'n_spills', '-')}")
print("\n" + SASS_HEADER)
for n in DIVE:
    save_ir(T[n], comp[n]["kernel"])

**Where the warps wait, by instruction (optional).** Nsight can sample each warp's program counter while the kernel
runs and record why the warp isn't issuing. Summing those samples per SASS instruction shows which instructions the
warps spend their time on: for v4, whether it is the barriers and shared-memory loads of the conversion, the reloads,
or the atomics. The cell prints, for v4, v11, v12 and v5, the opcodes and the single instructions holding the most
samples. It reads Nsight's source page; if your Nsight prints that page differently, the cell shows its first lines
instead, so paste them. Takes about 3 minutes.

In [ ]:
import csv, os, subprocess
from collections import Counter

OPCODE = re.compile(r"^\s*(?:@!?U?P\w+\s+)?([A-Z][A-Z0-9]*)")


def stall_samples(name):
    """[(SASS text, warp stall samples)] for one launch of `name` on normal, or (None, Nsight's output)."""
    os.makedirs("/content/pcsamp", exist_ok=True)
    rep = f"/content/pcsamp/{T[name]}"
    run = subprocess.run(["ncu", "--profile-from-start", "off", "--section", "SourceCounters", "--section",
                          "WarpStateStats", "-f", "-o", rep, "python", "prof.py", name, "normal"],
                         capture_output=True, text=True, cwd="/content")
    page = subprocess.run(["ncu", "--import", f"{rep}.ncu-rep", "--page", "source", "--print-source", "sass", "--csv"],
                          capture_output=True, text=True)
    lines = page.stdout.splitlines()
    start = next((i for i, l in enumerate(lines) if "Source" in l and "Address" in l), None)
    if start is None:
        return None, (run.stdout + run.stderr)[-1500:] + (page.stdout + page.stderr)[:1500]
    rows = list(csv.DictReader(lines[start:]))
    col = next((c for c in rows[0] if c.startswith("Warp Stall Sampling (All")), None) if rows else None
    if col is None:
        return None, "\n".join(lines[start:start + 5])
    return [(r["Source"].strip(), num(r[col])) for r in rows if r.get("Source")], None


for n in [V4, *lab.STRIDE_SPLIT, V5]:
    rows, err = stall_samples(n)
    if rows is None:
        print(f"{T[n]}: couldn't read Nsight's source page. First lines:\n{err}")
        break
    total = sum(s for _, s in rows if s == s) or 1
    ops = Counter()
    for src, s in rows:
        m = OPCODE.match(src)
        if m and s == s:
            ops[m[1]] += s
    print(f"{T[n]} ({n}), {total:,.0f} samples")
    print("  by opcode: " + ", ".join(f"{op} {100 * s / total:.1f}%" for op, s in ops.most_common(10)))
    for src, s in sorted((r for r in rows if r[1] == r[1]), key=lambda r: -r[1])[:6]:
        print(f"    {100 * s / total:5.1f}%  {src[:90]}")
    print()

## 8. Roofline

A roofline plots a kernel's achieved FLOP/s against its arithmetic intensity, the FLOPs it does per byte it moves
(Williams, Waterman and Patterson, 2009). The ceilings are the GPU's peak math rate (flat) and each memory level's
peak bandwidth (diagonal: bandwidth × intensity). A level's ridge point is where its diagonal meets the flat ceiling.
The hierarchical version plots one point per memory level (L1, L2, DRAM), all at the same height: a kernel has one
FLOP rate but moves a different number of bytes at each level.

What the first cell measures, on `normal`, under Nsight's locked clocks:
- **FLOPs:** tensor instructions executed (`sm__inst_executed_pipe_tensor_op_hmma`) × 4,096. Each
  `HMMA.16816.F32.BF16`, the only tensor instruction in both kernels' SASS, multiplies a 16×16 tile by a 16×8 tile:
  2·16·8·16 FLOP. The cell prints the count from the shape, 2·B·N·C·D = 137.4 GFLOP, beside it.
- **Bytes per level:** `l1tex__t_bytes`, `lts__t_bytes` and `dram__bytes`, the counters Nsight's own hierarchical
  roofline uses. `l1tex__t_bytes` counts global-memory traffic through L1, not shared-memory reads.
- **Bandwidth ceilings:** each counter's `peak_sustained` bytes per cycle × that level's measured clock.
- **Tensor ceiling:** 2,048 dense BF16 FLOP per SM per cycle (A100 whitepaper: 312 TFLOP/s = 108 SMs × 2,048 ×
  1.41 GHz) × the measured SM clock. Nsight locks the clock near 1.15 GHz, so the ceiling is about 255 TFLOP/s, not
  312. The cell checks this against Nsight's tensor busy %; the two should agree to within about a point.

The table's last rows turn each ceiling into time: how long the kernel would take if that one resource ran at its
peak and nothing else mattered. The resources work at the same time, so these times overlap and don't add up. The
largest is a lower bound on the runtime, and the gap between it and the measured duration is what the roofline
can't explain.

How to read it: a point left of its level's ridge is limited by that level's bandwidth. A point right of every
ridge but below the flat ceiling is limited by neither, and the gap is in how instructions issue and wait (sections
6 and 7). The roofline places the kernels; it doesn't explain the gap.

iter06 picks its launch settings by timing every candidate in each new process, and the Nsight process is a new
one. In the first run of this section its tuner picked a 256-key tile: each block reads all 2,048 of its batch's
centroids, so half as many blocks read half the centroid bytes, and the L1 and L2 points moved right for a reason
unrelated to the kernel's two changes. The cell now pins the settings section 7 recorded and the README reports
(`PIN`: BLOCK_N 128, BLOCK_K 64, 4 warps, 2 stages), and the table prints the grid and block size to confirm.

What to expect, from section 6's saved reports: about 87–88 MB of DRAM traffic for both kernels (about 1,570
FLOP/byte, far right of the DRAM ridge near 125), and 49% vs 70% of the tensor ceiling. The L2 and L1 points are new
here. iter06's atomics touch about 4× fewer sectors (section 7), but it also reloads its keys, so check which way its
L2 point moves.

Takes about 2 minutes and needs only the setup and `%%writefile` cells above. The first cell saves the numbers to
`MyDrive/kernel-agent/roofline.json`; the second draws the plot without rerunning Nsight and saves `roofline.png`.
To add kernels, put names from `lab.ablation()` in `ROOF`.

In [ ]:
import csv, json, subprocess
import torch
import triton
import lab

ROOF = ["original", "iter06"]  # add names from lab.ablation() to plot more kernels
PIN = {"iter06": "128,64,4,2,0"}  # the settings section 7 recorded and the README reports; see the note above
FLOP_PER_HMMA = 2 * 16 * 8 * 16  # HMMA.16816.F32.BF16, the only tensor instruction in both kernels' SASS (ir/)
TENSOR_FLOP_PER_SM_CYCLE = 2048  # A100 dense BF16: 312 TFLOP/s = 108 SMs x 2,048 x 1.41 GHz (A100 whitepaper)
LEVELS = {"L1": ("l1tex__t_bytes", "sm__cycles_elapsed"),  # bytes moved at each level, and the clock it runs on
          "L2": ("lts__t_bytes", "lts__cycles_elapsed"),
          "DRAM": ("dram__bytes", "dram__cycles_elapsed")}
REQUIRED = list(dict.fromkeys(["gpu__time_duration.sum", "launch__grid_size", "launch__block_size",
                               "sm__cycles_elapsed.avg.per_second"] + [
    m for count, clock in LEVELS.values() for m in (f"{count}.sum", f"{count}.sum.peak_sustained", f"{clock}.avg.per_second")]))
CHECKS = ["sm__inst_executed_pipe_tensor_op_hmma.sum",  # tensor instructions executed
          "sm__pipe_tensor_op_hmma_cycles_active.avg.pct_of_peak_sustained_elapsed"]  # Nsight's tensor busy %
SUFFIX = "" if triton.__version__ == "3.6.0" else f"-triton-{triton.__version__}"  # repo results/ and img/ = 3.6.0
PREFIX = {"": 1.0, "K": 1e3, "M": 1e6, "G": 1e9, "T": 1e12, "n": 1e-9, "u": 1e-6, "m": 1e-3}


def factor(unit):
    """Multiplier from an ncu unit (Mbyte, usecond, Ghz, byte/cycle, cycle/nsecond, ...) to bytes, cycles, seconds, Hz."""
    if "/" in unit:
        top, bottom = unit.split("/", 1)
        return factor(top) / factor(bottom)
    unit = {"s": "second", "ns": "nsecond", "us": "usecond", "ms": "msecond"}.get(unit.strip(), unit.strip())
    for base in ("byte", "cycle", "inst", "second", "hz"):
        if unit.lower().endswith(base):
            return PREFIX.get(unit[: -len(base)], 1.0)
    return 1.0


def ncu_raw(kernel, which, metrics):
    """{metric: value in base units} for the kernel's longest launch, or (None, ncu's output) if ncu failed."""
    cmd = ["ncu", "--profile-from-start", "off", "--csv", "--page", "raw", "--metrics", ",".join(metrics),
           "python", "prof.py", kernel, which] + ([PIN[kernel]] if kernel in PIN else [])
    out = subprocess.run(cmd, capture_output=True, text=True, cwd="/content")
    lines = out.stdout.splitlines()
    start = next((i for i, l in enumerate(lines) if l.startswith('"ID"')), None)
    if start is None:
        return None, out.stdout[-1500:] + out.stderr[-1500:]
    rows = list(csv.reader(lines[start:]))
    header = rows[0]
    units = next((dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and not r[0].strip()), {})
    launches = [dict(zip(header, r)) for r in rows[1:] if len(r) == len(header) and r[0].strip().isdigit()]

    def value(d, m):
        try:
            return float(d[m].replace(",", "")) * factor(units.get(m, ""))
        except (KeyError, ValueError):
            return float("nan")

    main = max(launches, key=lambda d: value(d, "gpu__time_duration.sum"))
    return {m: value(main, m) for m in metrics}, None


keys, cent = lab.inputs("normal")
(B, N, Dim), C = keys.shape, cent.shape[1]
FLOP_SHAPE = 2 * B * N * C * Dim  # one multiply-add per key, centroid and dimension
SMS = torch.cuda.get_device_properties(0).multi_processor_count

roof, roof_metrics = {}, REQUIRED + CHECKS
for kernel in ROOF:
    r, err = ncu_raw(kernel, "normal", roof_metrics)
    if r is None and roof_metrics != REQUIRED:
        print("ncu rejected a cross-check counter, retrying without them. ncu said:\n" + err)
        roof_metrics = REQUIRED
        r, err = ncu_raw(kernel, "normal", roof_metrics)
    if r is None:
        raise RuntimeError(err)
    flop_hmma = r.get(CHECKS[0], float("nan")) * FLOP_PER_HMMA
    flop = flop_hmma if flop_hmma == flop_hmma else FLOP_SHAPE  # the shape's count if the HMMA counter is missing
    roof[kernel] = {
        "seconds": r["gpu__time_duration.sum"],
        "grid": r["launch__grid_size"], "block": r["launch__block_size"],
        "flop": flop, "flop_hmma": flop_hmma, "flop_shape": FLOP_SHAPE,
        "flop_per_s": flop / r["gpu__time_duration.sum"],
        "tensor_peak": TENSOR_FLOP_PER_SM_CYCLE * SMS * r["sm__cycles_elapsed.avg.per_second"],
        "tensor_busy_pct": r.get(CHECKS[1], float("nan")),
        "bytes": {lv: r[f"{count}.sum"] for lv, (count, clock) in LEVELS.items()},
        "bandwidth": {lv: r[f"{count}.sum.peak_sustained"] * r[f"{clock}.avg.per_second"]
                      for lv, (count, clock) in LEVELS.items()},
        "raw": r,
    }
    print("profiled", kernel)

roof_rows = [
    ("Blocks x threads per block", lambda e: f"{e['grid']:,.0f} x {e['block']:.0f}"),
    ("Duration (us, clocks locked)", lambda e: f"{e['seconds'] * 1e6:,.0f}"),
    ("Tensor GFLOP, measured (HMMA x 4,096)", lambda e: f"{e['flop_hmma'] / 1e9:,.1f}"),
    ("Tensor GFLOP, from the shape (2BNCD)", lambda e: f"{e['flop_shape'] / 1e9:,.1f}"),
    ("Achieved TFLOP/s", lambda e: f"{e['flop_per_s'] / 1e12:,.1f}"),
    ("% of BF16 tensor ceiling", lambda e: f"{100 * e['flop_per_s'] / e['tensor_peak']:.1f}"),
    ("  Nsight tensor busy % (cross-check)", lambda e: f"{e['tensor_busy_pct']:.1f}"),
    *[(f"{lv} MB moved", lambda e, lv=lv: f"{e['bytes'][lv] / 1e6:,.1f}") for lv in LEVELS],
    *[(f"{lv} FLOP/byte", lambda e, lv=lv: f"{e['flop'] / e['bytes'][lv]:,.0f}") for lv in LEVELS],
    ("Time at peak rate, each alone (us):", lambda e: ""),
    ("  BF16 tensor", lambda e: f"{e['flop'] / e['tensor_peak'] * 1e6:,.0f}"),
    *[(f"  {lv}", lambda e, lv=lv: f"{e['bytes'][lv] / e['bandwidth'][lv] * 1e6:,.0f}") for lv in LEVELS],
    ("  measured duration", lambda e: f"{e['seconds'] * 1e6:,.0f}"),
]
print(f"\n{'':40s}" + "".join(f"{k:>12s}" for k in roof))
for label, cell in roof_rows:
    print(f"{label:40s}" + "".join(f"{cell(e):>12s}" for e in roof.values()))

peaks = next(iter(roof.values()))  # clocks are locked, so every kernel sees the same ceilings
print(f"\nceilings at {peaks['raw']['sm__cycles_elapsed.avg.per_second'] / 1e9:.3f} GHz (SM), {SMS} SMs:")
print(f"  BF16 tensor  {peaks['tensor_peak'] / 1e12:7,.1f} TFLOP/s")
for lv, bw in peaks["bandwidth"].items():
    print(f"  {lv:11s}  {bw / 1e9:7,.0f} GB/s   ridge {peaks['tensor_peak'] / bw:6,.1f} FLOP/byte")

with open(f"{D}/roofline{SUFFIX}.json", "w") as f:
    json.dump(roof, f, indent=1)
print(f"\nsaved {D}/roofline{SUFFIX}.json")

In [ ]:
import json, math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import triton

SUFFIX = "" if triton.__version__ == "3.6.0" else f"-triton-{triton.__version__}"  # repo results/ and img/ = 3.6.0
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # one per kernel, in this fixed order; past three, plot separately
MARKERS = {"L1": dict(marker="o", ms=13, mfc="none", mew=2),  # an open ring, visible around an L2 square at the same spot
           "L2": dict(marker="s", ms=8, mec=SURFACE, mew=1.5),
           "DRAM": dict(marker="^", ms=9, mec=SURFACE, mew=1.5)}

if "roof" not in globals():  # a new session: redraw from the saved numbers without rerunning Nsight
    roof = json.load(open(f"{D}/roofline{SUFFIX}.json"))
peaks = next(iter(roof.values()))
ceiling = peaks["tensor_peak"] / 1e12  # TFLOP/s
bw = {lv: b / 1e12 for lv, b in peaks["bandwidth"].items()}  # TB/s, so TFLOP/s = FLOP/byte x TB/s
ai = {k: {lv: e["flop"] / e["bytes"][lv] for lv in MARKERS} for k, e in roof.items()}
x_lo = min([ceiling / b for b in bw.values()] + [a for d in ai.values() for a in d.values()]) / 8
x_hi = max(a for d in ai.values() for a in d.values()) * 10  # room for the kernel labels right of the points
y_lo = min(b * x_lo for b in bw.values()) / 2

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=120, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.set(xscale="log", yscale="log", xlim=(x_lo, x_hi), ylim=(y_lo, ceiling * 2.5))
xs = [x_lo * (x_hi / x_lo) ** (i / 200) for i in range(201)]
for b in bw.values():
    ax.plot(xs, [min(ceiling, b * x) for x in xs], color=MUTED, lw=1.5, zorder=1)

for (k, e), color in zip(roof.items(), COLORS):
    y = e["flop_per_s"] / 1e12
    ax.plot([min(ai[k].values()), max(ai[k].values())], [y, y], color=color, lw=2, zorder=2)
    for lv in MARKERS:
        ax.plot(ai[k][lv], y, ls="", color=color, zorder=3, **MARKERS[lv])
    ax.annotate(f"{k}: {y:.0f} TFLOP/s ({100 * e['flop_per_s'] / e['tensor_peak']:.0f}%)",
                (max(ai[k].values()), y), xytext=(10, 0), textcoords="offset points",
                va="center", color=INK, fontsize=9)

ticks = FuncFormatter(lambda v, _: f"{v:,.0f}" if v >= 1 else f"{v:g}")
ax.xaxis.set_major_formatter(ticks)
ax.yaxis.set_major_formatter(ticks)
ax.grid(True, which="major", color=GRID, lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(MUTED)
ax.tick_params(colors=MUTED, which="both")
ax.set_xlabel("Arithmetic intensity (FLOP per byte moved at that memory level)", color=INK)
ax.set_ylabel("BF16 tensor TFLOP/s", color=INK)
ax.set_title(f"Hierarchical roofline, A100, Triton {triton.__version__} (Nsight Compute, locked clocks, keys: normal)",
             color=INK, loc="left", fontsize=11)
legend_items = [Line2D([], [], color=c, lw=2, label=k) for k, c in zip(roof, COLORS)]
legend_items += [Line2D([], [], ls="", color=MUTED, label=f"{lv} bytes", **MARKERS[lv]) for lv in MARKERS]
ax.legend(handles=legend_items, loc="lower right", frameon=False, fontsize=9, labelcolor=INK)
fig.tight_layout()

# Ceiling labels go on last: a diagonal's angle on screen depends on the final axes size.
(x0, y0), (x1, y1) = ax.transData.transform([(x_lo, x_lo), (x_lo * 10, x_lo * 10)])
slope = math.degrees(math.atan2(y1 - y0, x1 - x0))
for lv, b in bw.items():
    x = ceiling / b / 6
    ax.text(x, b * x * 1.15, f"{lv} {b:,.1f} TB/s", rotation=slope, rotation_mode="anchor", color=MUTED, fontsize=9)
ax.text(x_hi / 1.1, ceiling * 1.12, f"BF16 tensor {ceiling:,.0f} TFLOP/s at "
        f"{peaks['raw']['sm__cycles_elapsed.avg.per_second'] / 1e9:.2f} GHz", ha="right", va="bottom",
        color=MUTED, fontsize=9)

fig.savefig(f"{D}/roofline{SUFFIX}.png", dpi=150, bbox_inches="tight", facecolor=SURFACE)
plt.show()
print(f"saved {D}/roofline{SUFFIX}.png")

## 9. Checklist for the bullet

| Claim | Where to check | Earlier result |
|---|---|---|
| iter06 about 1.36× on real keys | section 4, per-row summary | old files' 14 ordinary rows: 1.35–1.38× |
| iter06 passes the strict test on real and held-out keys | section 3 | emulated worst cos gap 1.9e-4 |
| a third pass from iter06 found nothing faster | `kernel_agent.ipynb` step 6 log | 6 iterations, none better |
| int8 1.74× on synthetic keys | section 4, whole batches | 1.737× |
| int8 passes phase 1's test on synthetic keys, fails the strict test | section 3 | emulated: fails 3 of 16 real rows and synthetic |
| phase 2 is slower than iter06 on real keys | section 4 | old ordinary rows 1.25–1.28× |
| which change buys what | section 4, ablation | (new here) |
| what the rest of the gap between variant 5 and iter06 is | section 7, variants 6 and 7 | (new here) |
| which change cuts atomic sectors per request, and how | section 7, Nsight per variant and layout replay | (new here) |
| whether the hidden stride alone fixes the tile atomics | section 7, v8 and v9 | (new here) |
| where both kernels sit against the A100's ceilings | section 8, roofline | (new here) |
| whether packing is needed for the instruction savings | section 7, v10 | (new here) |
| what makes v4 slower than v5 | section 7, v4 vs v5 | (new here) |
| clock drift between processes | section 5 | ~18% in the first runs |

Draft bullets (update the numbers from your outputs):
- *Optimized the Triton k-means assignment kernel from my sparse-attention paper (Double-P, arXiv 2602.05191) with a
  Claude-driven agent loop guided by Nsight Compute: **1.36× faster on real Qwen2.5-1.5B attention keys** (coalesced
  atomic adds, packed score/index argmax). Assignment error is bounded at 2^-12 in cosine similarity (16× finer than
  the bf16 inputs) and validated on held-out keys.*
- *Built the evaluation harness that made the results trustworthy: same-process interleaved timing, a real-key
  correctness gate with held-out keys, and a repeat-call check. It **rejected an int8 kernel (1.74×) that passed
  synthetic tests but picked wrong centroids**. Traced an apparent 2.4× to benchmark keys that skipped the
  pipeline's mean-centering, rebuilt them to match production, and confirmed with a further agent pass that the
  1.36× kernel was the best found.*